# Step 1: Create one combined Lightcast input file

In [11]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# ============================================================
# PATHS
# ============================================================

LIGHTCAST_FILE = Path(
    r"C:\Users\301533\Documents\AI Analysis\Ligthcast Skills & RO\Lightcast_AI_High_AND_VeryHigh_Compatible_Occs_CLEANED.xlsx"
)

LIGHTCAST_VERY_HIGH_FOLDER = Path(
    r"C:\Users\301533\Documents\AI Analysis\Ligthcast Skills & RO"
)

LIGHTCAST_HIGH_FOLDER = Path(
    r"C:\Users\301533\Documents\AI Analysis\Ligthcast Skills & RO\High AI"
)

ONET_FILE = Path(
    r"C:\Users\301533\Documents\Green Analysis\Related Occupations.xlsx"
)

CROSSWALK_FILE = Path(
    r"C:\Users\301533\Documents\AI Analysis\2019_to_SOC_Crosswalk.xlsx"
)

AI_FILE = Path(
    r"G:\Shared drives\EO_ECONOMIC ANALYSIS\ea_common\Projects\AI\risk_index\ai_exposure_composite_clean.csv"
)

OUTPUT_FILE = Path(
    r"C:\Users\301533\Documents\AI Analysis\Ligthcast Skills & RO\FINAL_High_AND_VeryHigh_AI_Lightcast_ONET_Transitions_WITH_GATEWAYS.xlsx"
)

SOURCE_EXPOSURE_GROUPS = ["High", "Very High"]
LOWER_EXPOSURE_GROUPS = ["Very Low", "Low", "Medium"]

# ============================================================
# HELPERS
# ============================================================

month_map = {
    "Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04",
    "May": "05", "Jun": "06", "Jul": "07", "Aug": "08",
    "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"
}

def clean_title(x):
    if pd.isna(x):
        return np.nan
    return (
        str(x)
        .strip()
        .lower()
        .replace("*", "")
        .replace(",", "")
        .replace("  ", " ")
    )

def fix_soc(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().replace(".0", "")

    if x.lower() in ["nan", "none", ""]:
        return np.nan

    match = re.search(r"(\d{2})-(\d{4})", x)
    if match:
        return f"{match.group(1)}-{match.group(2)}"

    if "-" in x:
        left, right = x.split("-", 1)

        if left in month_map and right.isdigit():
            if len(right) <= 2:
                return f"{month_map[left]}-20{right.zfill(2)}"
            return f"{month_map[left]}-{right.zfill(4)}"

        if right in month_map and left.isdigit():
            return f"{month_map[right]}-20{left.zfill(2)}"

    return x

def clean_number(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    if s in ["", "-", "NA", "N/A", "nan", "None"]:
        return np.nan

    s = (
        s.replace("$", "")
         .replace("/hr", "")
         .replace(",", "")
         .replace("%", "")
         .replace("−", "-")
    )

    return pd.to_numeric(s, errors="coerce")

def extract_occ_from_filename(file):
    name = file.stem
    name = name.replace("Skills_Transferability_", "")
    name = re.sub(r"_in_(United_States|Arizona).*", "", name)
    return name.replace("_", " ").strip()

def extract_soc_from_text(text):
    if not isinstance(text, str):
        return np.nan

    match = re.search(r"SOC\s+(\d{2}-\d{4})", text)
    if match:
        return fix_soc(match.group(1))

    match = re.search(r"(\d{2}-\d{4})", text)
    if match:
        return fix_soc(match.group(1))

    return np.nan

def get_soc_from_overview(file):
    try:
        temp = pd.read_excel(file, sheet_name="Overview", header=None)

        for value in temp.astype(str).values.flatten():
            soc = extract_soc_from_text(value)
            if pd.notna(soc):
                return soc

    except Exception:
        return np.nan

    return np.nan

# ============================================================
# LOAD CROSSWALK
# ============================================================

crosswalk = pd.read_excel(CROSSWALK_FILE, header=3)
crosswalk.columns = crosswalk.columns.astype(str).str.strip()

crosswalk = crosswalk.rename(columns={
    "O*NET-SOC 2019 Code": "ONET_2019_Code",
    "O*NET-SOC 2019 Title": "ONET_2019_Title",
    "2018 SOC Code": "SOC_2018",
    "2018 SOC Title": "SOC_2018_Title"
})

crosswalk["ONET_2019_Code_Clean"] = crosswalk["ONET_2019_Code"].apply(fix_soc)
crosswalk["SOC_2018_Clean"] = crosswalk["SOC_2018"].apply(fix_soc)
crosswalk["Title_Clean"] = crosswalk["ONET_2019_Title"].apply(clean_title)

title_to_soc2018 = (
    crosswalk
    .dropna(subset=["Title_Clean", "SOC_2018_Clean"])
    .drop_duplicates(subset=["Title_Clean"])
    .set_index("Title_Clean")["SOC_2018_Clean"]
    .to_dict()
)

soc_title_lookup = (
    crosswalk[["SOC_2018_Clean", "SOC_2018_Title"]]
    .dropna()
    .drop_duplicates(subset=["SOC_2018_Clean"])
    .copy()
)

soc_title_dict = dict(
    zip(soc_title_lookup["SOC_2018_Clean"], soc_title_lookup["SOC_2018_Title"])
)

print("Loaded crosswalk rows:", crosswalk.shape)

# ============================================================
# LOAD AI COMPOSITE
# ============================================================

ai = pd.read_csv(AI_FILE)
ai.columns = ai.columns.astype(str).str.strip()

possible_soc_cols = [
    "SOC", "SOC Code", "SOC_Code", "SOC2018", "SOC2018.1",
    "O*NET-SOC Code", "O*NET_SOC_Code", "O*NET-SOC Code.1"
]

possible_title_cols = [
    "Title", "Occupation", "Occupation Title",
    "2018 SOC Title", "O*NET-SOC 2019 Title"
]

possible_exposure_cols = [
    "AI Exposure Category", "AI_Exposure_Group", "exposure", "Exposure",
    "AI Exposure Group", "AI_Exposure_Category", "Exposure_Group"
]

soc_col = next((c for c in possible_soc_cols if c in ai.columns), None)
title_col = next((c for c in possible_title_cols if c in ai.columns), None)
exposure_col = next((c for c in possible_exposure_cols if c in ai.columns), None)

if soc_col is None:
    raise ValueError("Could not find SOC column in AI file.")

if title_col is None:
    raise ValueError("Could not find occupation title column in AI file.")

if exposure_col is None:
    raise ValueError("Could not find exposure column in AI file.")

ai["Title_Clean"] = ai[title_col].apply(clean_title)
ai["SOC"] = ai["Title_Clean"].map(title_to_soc2018)
ai["SOC_Fallback"] = ai[soc_col].apply(fix_soc)
ai["SOC"] = ai["SOC"].fillna(ai["SOC_Fallback"])
ai["AI_Exposure_Group"] = ai[exposure_col].astype(str).str.strip()

ai_lookup = (
    ai[["SOC", "AI_Exposure_Group"]]
    .dropna()
    .drop_duplicates(subset=["SOC"])
    .copy()
)

source_socs = set(
    ai_lookup.loc[
        ai_lookup["AI_Exposure_Group"].isin(SOURCE_EXPOSURE_GROUPS),
        "SOC"
    ]
)

print("Using SOC column:", soc_col)
print("Using title column:", title_col)
print("Using exposure column:", exposure_col)
print("High + Very High SOCs from composite file:", len(source_socs))

# ============================================================
# BUILD AI COMPOSITE ECONOMIC SHEET
# ============================================================

ai_composite_economic = ai.copy()
ai_composite_economic["SOC"] = ai_composite_economic["SOC"].apply(fix_soc)

needed_ai_cols = [
    "SOC",
    title_col,
    exposure_col,
    "Annual Total Openings",
    "Annual Percent Change",
    "Education Value",
    "Work Experience Value",
    "Growth Rate",
    "50th Percentile Wage"
]

needed_ai_cols = [c for c in needed_ai_cols if c in ai_composite_economic.columns]

ai_composite_economic = ai_composite_economic[needed_ai_cols].copy()

rename_ai_cols = {
    title_col: "Occupation",
    exposure_col: "AI_Exposure_Group",
    "Annual Total Openings": "AI_Composite_Annual_Total_Openings",
    "Annual Percent Change": "AI_Composite_Annual_Percent_Change",
    "Education Value": "AI_Composite_Education_Value",
    "Work Experience Value": "AI_Composite_Work_Experience_Value",
    "Growth Rate": "AI_Composite_Growth_Rate",
    "50th Percentile Wage": "AI_Composite_50th_Percentile_Wage"
}

ai_composite_economic = ai_composite_economic.rename(columns=rename_ai_cols)
ai_composite_economic = ai_composite_economic.drop_duplicates(subset=["SOC"])

# ============================================================
# LOAD LIGHTCAST CLEANED COMPATIBLE OCCUPATIONS
# ============================================================

xls = pd.ExcelFile(LIGHTCAST_FILE)

if "Cleaned_Compatible_Occs" in xls.sheet_names:
    lightcast_sheet = "Cleaned_Compatible_Occs"
else:
    lightcast_sheet = xls.sheet_names[0]

print("Using Lightcast sheet:", lightcast_sheet)

lc = pd.read_excel(LIGHTCAST_FILE, sheet_name=lightcast_sheet)
lc.columns = lc.columns.astype(str).str.strip()

lc["Source_SOC"] = lc["Source_SOC"].apply(fix_soc)
lc["Target_SOC"] = lc["Target_SOC"].apply(fix_soc)

lc = lc[lc["Source_SOC"].isin(source_socs)].copy()

lc["pair_key"] = (
    lc["Source_SOC"].astype(str) + "_" + lc["Target_SOC"].astype(str)
)

print("Lightcast rows after High + Very High source filter:", lc.shape)

# ============================================================
# LOAD O*NET RELATED OCCUPATIONS
# ============================================================

onet = pd.read_excel(ONET_FILE)
onet.columns = onet.columns.astype(str).str.strip()

onet = onet.rename(columns={
    "O*NET-SOC Code": "Base_SOC",
    "Title": "Base_Title",
    "Related O*NET-SOC Code": "Related_SOC",
    "Related Title": "Related_Title",
    "Relatedness Tier": "ONET_Tier",
    "Index": "ONET_Index"
})

onet["Base_SOC"] = onet["Base_SOC"].apply(fix_soc)
onet["Related_SOC"] = onet["Related_SOC"].apply(fix_soc)

onet = onet.dropna(subset=["Base_SOC", "Related_SOC"]).copy()
onet = onet[onet["Base_SOC"].isin(source_socs)].copy()

tier_priority = {
    "Primary-Short": 1,
    "Primary-Long": 2,
    "Supplemental": 3
}

onet["ONET_Tier_Priority"] = onet["ONET_Tier"].map(tier_priority)

onet = (
    onet.sort_values(
        ["Base_SOC", "Related_SOC", "ONET_Tier_Priority", "ONET_Index"]
    )
    .drop_duplicates(subset=["Base_SOC", "Related_SOC"], keep="first")
    .copy()
)

onet["pair_key"] = (
    onet["Base_SOC"].astype(str) + "_" + onet["Related_SOC"].astype(str)
)

print("O*NET rows after High + Very High source filter and collapse:", onet.shape)

# ============================================================
# MERGE LIGHTCAST WITH O*NET
# ============================================================

merged = lc.merge(
    onet[
        [
            "Base_SOC",
            "Related_SOC",
            "ONET_Tier",
            "ONET_Index",
            "Base_Title",
            "Related_Title",
            "pair_key"
        ]
    ],
    left_on=["Source_SOC", "Target_SOC"],
    right_on=["Base_SOC", "Related_SOC"],
    how="left",
    suffixes=("", "_ONET")
)

merged["Source_Type"] = np.where(
    merged["ONET_Tier"].notna(),
    "Both",
    "Lightcast"
)

merged["pair_key"] = (
    merged["Source_SOC"].astype(str) + "_" + merged["Target_SOC"].astype(str)
)

# ============================================================
# CREATE O*NET ONLY ROWS
# ============================================================

lightcast_pair_keys = set(merged["pair_key"].dropna())
onet_only = onet[~onet["pair_key"].isin(lightcast_pair_keys)].copy()

print("O*NET-only rows to append:", onet_only.shape)

onet_only_formatted = pd.DataFrame({
    "Source_SOC": onet_only["Base_SOC"],
    "Source_Occupation": onet_only["Base_Title"],
    "Compat_Rank": np.nan,
    "Target_SOC": onet_only["Related_SOC"],
    "Target_Occupation": onet_only["Related_Title"],
    "Final_Compat_Index": np.nan,
    "Weighted_Compat_Index": np.nan,
    "Avg_Compat_Index": np.nan,
    "Min_Compat_Index": np.nan,
    "Max_Compat_Index": np.nan,
    "Median_Hourly_Earnings": np.nan,
    "Jobs_2020": np.nan,
    "Jobs_2025": np.nan,
    "Change_2020_2025": np.nan,
    "Estimated_Annual_Openings": np.nan,
    "Source_Row_Count": np.nan,
    "Target_SOC_Detailed_Options": onet_only["Related_SOC"],
    "Target_Occupation_Options": onet_only["Related_Title"],
    "ONET_Tier": onet_only["ONET_Tier"],
    "ONET_Index": onet_only["ONET_Index"],
    "Base_Title": onet_only["Base_Title"],
    "Related_Title": onet_only["Related_Title"],
    "Source_Type": "ONET",
    "pair_key": onet_only["pair_key"]
})

for col in merged.columns:
    if col not in onet_only_formatted.columns:
        onet_only_formatted[col] = np.nan

onet_only_formatted = onet_only_formatted[merged.columns]

# ============================================================
# FINAL TRANSITIONS
# ============================================================

final = pd.concat(
    [merged, onet_only_formatted],
    ignore_index=True
)

final["Source_Occupation_Original"] = final["Source_Occupation"]
final["Target_Occupation_Original"] = final["Target_Occupation"]

final["Source_Occupation"] = (
    final["Source_SOC"]
    .map(soc_title_dict)
    .fillna(final["Source_Occupation"])
)

final["Target_Occupation"] = (
    final["Target_SOC"]
    .map(soc_title_dict)
    .fillna(final["Target_Occupation"])
)

# ============================================================
# MERGE AI EXPOSURE FOR SOURCE AND TARGET
# ============================================================

final = final.merge(
    ai_lookup.rename(columns={
        "SOC": "Source_SOC",
        "AI_Exposure_Group": "Source_AI_Exposure_Group_LOOKUP"
    }),
    on="Source_SOC",
    how="left"
)

final = final.merge(
    ai_lookup.rename(columns={
        "SOC": "Target_SOC",
        "AI_Exposure_Group": "Target_AI_Exposure_Group_LOOKUP"
    }),
    on="Target_SOC",
    how="left"
)

if "Source_AI_Exposure_Group" in final.columns:
    final["Source_AI_Exposure_Group"] = (
        final["Source_AI_Exposure_Group_LOOKUP"]
        .fillna(final["Source_AI_Exposure_Group"])
    )
else:
    final["Source_AI_Exposure_Group"] = final["Source_AI_Exposure_Group_LOOKUP"]

if "Target_AI_Exposure_Group" in final.columns:
    final["Target_AI_Exposure_Group"] = (
        final["Target_AI_Exposure_Group_LOOKUP"]
        .fillna(final["Target_AI_Exposure_Group"])
    )
else:
    final["Target_AI_Exposure_Group"] = final["Target_AI_Exposure_Group_LOOKUP"]

final = final.drop(
    columns=[
        "Source_AI_Exposure_Group_LOOKUP",
        "Target_AI_Exposure_Group_LOOKUP"
    ],
    errors="ignore"
)

final = final[
    final["Source_AI_Exposure_Group"].isin(SOURCE_EXPOSURE_GROUPS)
].copy()

# ============================================================
# BUILD LIGHTCAST ECONOMIC / EDUCATION / KSA SHEETS
# ============================================================

lightcast_files = list(LIGHTCAST_VERY_HIGH_FOLDER.glob("Skills_Transferability_*.xlsx"))
lightcast_files += list(LIGHTCAST_HIGH_FOLDER.glob("Skills_Transferability_*.xlsx"))

overview_rows = []
education_rows = []
ksa_rows = []
lightcast_errors = []

for file in lightcast_files:
    occupation = extract_occ_from_filename(file)
    soc_code = get_soc_from_overview(file)

    try:
        xls_file = pd.ExcelFile(file)

        if "Overview" in xls_file.sheet_names:
            temp = pd.read_excel(file, sheet_name="Overview", header=None)

            metrics = {}

            for _, row in temp.iterrows():
                label = row.iloc[0] if len(row) > 0 else np.nan
                value = row.iloc[1] if len(row) > 1 else np.nan

                if isinstance(label, str):
                    label_clean = label.strip()

                    if label_clean in [
                        "Median Hourly Earnings*",
                        "2025 Jobs*",
                        "2024-2025 Change*",
                        "2024-2025 Estimated Annual Openings*"
                    ]:
                        metrics[label_clean] = value

            median_hourly = clean_number(metrics.get("Median Hourly Earnings*"))
            median_annual = median_hourly * 2080 if pd.notna(median_hourly) else np.nan

            overview_rows.append({
                "SOC": soc_code,
                "Occupation": occupation,
                "Lightcast_Median_Hourly_Earnings": median_hourly,
                "Lightcast_Median_Annual_Earnings": median_annual,
                "Lightcast_2025_Jobs": clean_number(metrics.get("2025 Jobs*")),
                "Lightcast_2024_2025_Change": clean_number(metrics.get("2024-2025 Change*")),
                "Lightcast_2024_2025_Estimated_Annual_Openings": clean_number(metrics.get("2024-2025 Estimated Annual Openings*")),
                "Source_File": file.name
            })

        if "Edu Attainment" in xls_file.sheet_names:
            temp = pd.read_excel(file, sheet_name="Edu Attainment", header=None)
            temp = temp.dropna(how="all")

            for _, row in temp.iterrows():
                education = row.iloc[0] if len(row) > 0 else np.nan
                percent = row.iloc[1] if len(row) > 1 else np.nan

                if isinstance(education, str):
                    education = education.strip()

                    if education not in ["Educational Attainment", ""]:
                        pct = clean_number(percent)

                        education_rows.append({
                            "SOC": soc_code,
                            "Occupation": occupation,
                            "Education_Level": education,
                            "Education_Percent": pct / 100 if pd.notna(pct) and pct > 1 else pct,
                            "Source_File": file.name
                        })

        ksa_tabs = {
            "Competencies - Knowledge": "Knowledge",
            "Competencies - Skills": "Skill",
            "Competencies - Abilities": "Ability"
        }

        for sheet, category in ksa_tabs.items():
            if sheet in xls_file.sheet_names:
                temp = pd.read_excel(file, sheet_name=sheet, header=2)
                temp = temp.dropna(how="all")
                temp.columns = temp.columns.astype(str).str.strip()

                for _, row in temp.iterrows():
                    ksa_name = row.get(category)

                    if pd.isna(ksa_name) or str(ksa_name).strip() == "":
                        continue

                    ksa_rows.append({
                        "SOC": soc_code,
                        "Occupation": occupation,
                        "KSA_Category": category,
                        "KSA_Name": str(ksa_name).strip(),
                        "Importance": clean_number(row.get("Importance")),
                        "Level": clean_number(row.get("Level")),
                        "Source_File": file.name
                    })

    except Exception as e:
        lightcast_errors.append({
            "File": file.name,
            "Occupation": occupation,
            "SOC": soc_code,
            "Error": str(e)
        })

lightcast_economic = pd.DataFrame(overview_rows)
lightcast_education = pd.DataFrame(education_rows)
lightcast_ksas = pd.DataFrame(ksa_rows)
lightcast_errors = pd.DataFrame(lightcast_errors)

if not lightcast_education.empty:
    lightcast_top_education = (
        lightcast_education
        .sort_values(["SOC", "Education_Percent"], ascending=[True, False])
        .drop_duplicates(subset=["SOC"])
        .rename(columns={
            "Education_Level": "Lightcast_Top_Education_Level",
            "Education_Percent": "Lightcast_Top_Education_Percent"
        })
        [["SOC", "Lightcast_Top_Education_Level", "Lightcast_Top_Education_Percent"]]
    )
else:
    lightcast_top_education = pd.DataFrame(columns=[
        "SOC", "Lightcast_Top_Education_Level", "Lightcast_Top_Education_Percent"
    ])

# ============================================================
# MERGE LIGHTCAST AND AI COMPOSITE DATA INTO FINAL
# ============================================================

if not lightcast_economic.empty:
    final = final.merge(
        lightcast_economic.rename(columns={"SOC": "Target_SOC"}),
        on="Target_SOC",
        how="left"
    )

if not lightcast_top_education.empty:
    final = final.merge(
        lightcast_top_education.rename(columns={"SOC": "Target_SOC"}),
        on="Target_SOC",
        how="left"
    )

final = final.merge(
    ai_composite_economic.rename(columns={
        "SOC": "Target_SOC",
        "Occupation": "AI_Target_Occupation"
    }),
    on="Target_SOC",
    how="left"
)

# ============================================================
# CLEANUP FINAL TRANSITIONS
# ============================================================

final = final.drop(
    columns=[
        "Base_SOC",
        "Related_SOC",
        "pair_key",
        "pair_key_ONET",
        "ONET_Tier_Priority",
        "Transition_Confidence"
    ],
    errors="ignore"
)

final = (
    final.sort_values(
        ["Source_SOC", "Source_Type", "Compat_Rank", "ONET_Index", "Target_SOC"],
        na_position="last"
    )
    .drop_duplicates(subset=["Source_SOC", "Target_SOC"], keep="first")
    .copy()
)

# ============================================================
# BUILD GATEWAY PATHWAYS
# ============================================================

gateway_base = final.copy()

gateway_base = gateway_base[
    gateway_base["Source_AI_Exposure_Group"].isin(SOURCE_EXPOSURE_GROUPS)
].copy()

gateway_base = gateway_base[
    gateway_base["Target_AI_Exposure_Group"].isin(LOWER_EXPOSURE_GROUPS)
].copy()

gateway_base["Final_Compat_Index"] = pd.to_numeric(
    gateway_base["Final_Compat_Index"], errors="coerce"
)

gateway_base["Jobs_2025"] = pd.to_numeric(
    gateway_base["Jobs_2025"], errors="coerce"
)

step1 = (
    gateway_base
    .sort_values(
        ["Source_SOC", "Final_Compat_Index", "Jobs_2025"],
        ascending=[True, False, False],
        na_position="last"
    )
    .groupby("Source_SOC")
    .head(3)
    .copy()
)

step1["Gateway_Rank"] = step1.groupby("Source_SOC").cumcount() + 1

step1 = step1.rename(columns={
    "Source_SOC": "Original_Source_SOC",
    "Source_Occupation": "Original_Source_Occupation",
    "Source_AI_Exposure_Group": "Original_Source_AI_Exposure_Group",
    "Target_SOC": "Gateway_SOC",
    "Target_Occupation": "Gateway_Occupation",
    "Target_AI_Exposure_Group": "Gateway_AI_Exposure_Group",
    "Source_Type": "Gateway_Source_Type",
    "ONET_Tier": "Gateway_ONET_Tier",
    "Final_Compat_Index": "Gateway_Compat_Index",
    "Median_Hourly_Earnings": "Gateway_Median_Hourly_Earnings",
    "Jobs_2025": "Gateway_Jobs_2025",
    "Estimated_Annual_Openings": "Gateway_Estimated_Annual_Openings"
})

onet_dest = pd.read_excel(ONET_FILE)
onet_dest.columns = onet_dest.columns.astype(str).str.strip()

onet_dest = onet_dest.rename(columns={
    "O*NET-SOC Code": "Gateway_SOC",
    "Title": "Gateway_Title_Check",
    "Related O*NET-SOC Code": "Destination_SOC",
    "Related Title": "Destination_Occupation_Original",
    "Relatedness Tier": "Destination_ONET_Tier",
    "Index": "Destination_ONET_Index"
})

onet_dest["Gateway_SOC"] = onet_dest["Gateway_SOC"].apply(fix_soc)
onet_dest["Destination_SOC"] = onet_dest["Destination_SOC"].apply(fix_soc)

onet_dest = onet_dest.dropna(subset=["Gateway_SOC", "Destination_SOC"]).copy()

onet_dest["Destination_Occupation"] = (
    onet_dest["Destination_SOC"]
    .map(soc_title_dict)
    .fillna(onet_dest["Destination_Occupation_Original"])
)

onet_dest = onet_dest.merge(
    ai_lookup.rename(columns={
        "SOC": "Destination_SOC",
        "AI_Exposure_Group": "Destination_AI_Exposure_Group"
    }),
    on="Destination_SOC",
    how="left"
)

onet_dest = onet_dest[
    onet_dest["Destination_AI_Exposure_Group"].isin(LOWER_EXPOSURE_GROUPS)
].copy()

onet_dest["Destination_ONET_Tier_Priority"] = (
    onet_dest["Destination_ONET_Tier"].map(tier_priority)
)

onet_dest = (
    onet_dest
    .sort_values(
        [
            "Gateway_SOC",
            "Destination_SOC",
            "Destination_ONET_Tier_Priority",
            "Destination_ONET_Index"
        ]
    )
    .drop_duplicates(subset=["Gateway_SOC", "Destination_SOC"], keep="first")
    .copy()
)

gateway_paths = step1.merge(
    onet_dest,
    on="Gateway_SOC",
    how="left"
)

gateway_paths = gateway_paths[gateway_paths["Destination_SOC"].notna()].copy()

gateway_paths = gateway_paths[
    gateway_paths["Destination_SOC"] != gateway_paths["Original_Source_SOC"]
].copy()

gateway_paths = gateway_paths[
    gateway_paths["Destination_SOC"] != gateway_paths["Gateway_SOC"]
].copy()

gateway_paths["Pathway_Type"] = "Gateway Pathway"

gateway_paths["Pathway_Label"] = (
    gateway_paths["Original_Source_Occupation"].astype(str)
    + " -> "
    + gateway_paths["Gateway_Occupation"].astype(str)
    + " -> "
    + gateway_paths["Destination_Occupation"].astype(str)
)

gateway_paths = gateway_paths.sort_values(
    [
        "Original_Source_SOC",
        "Gateway_Rank",
        "Gateway_SOC",
        "Destination_ONET_Tier_Priority",
        "Destination_ONET_Index"
    ],
    ascending=[True, True, True, True, True],
    na_position="last"
).copy()

gateway_cols = [
    "Pathway_Type",
    "Pathway_Label",
    "Original_Source_SOC",
    "Original_Source_Occupation",
    "Original_Source_AI_Exposure_Group",
    "Gateway_Rank",
    "Gateway_SOC",
    "Gateway_Occupation",
    "Gateway_AI_Exposure_Group",
    "Gateway_Source_Type",
    "Gateway_ONET_Tier",
    "Gateway_Compat_Index",
    "Gateway_Median_Hourly_Earnings",
    "Gateway_Jobs_2025",
    "Gateway_Estimated_Annual_Openings",
    "Destination_SOC",
    "Destination_Occupation",
    "Destination_AI_Exposure_Group",
    "Destination_ONET_Tier",
    "Destination_ONET_Index"
]

gateway_cols = [c for c in gateway_cols if c in gateway_paths.columns]
gateway_paths = gateway_paths[gateway_cols].copy()

# ============================================================
# CHECKS
# ============================================================

duplicate_check = final[
    final.duplicated(subset=["Source_SOC", "Target_SOC"], keep=False)
].copy()

source_title_check = (
    final.groupby("Source_SOC")["Source_Occupation"]
    .nunique()
    .reset_index(name="Unique_Source_Title_Count")
    .sort_values("Unique_Source_Title_Count", ascending=False)
)

target_title_check = (
    final.groupby("Target_SOC")["Target_Occupation"]
    .nunique()
    .reset_index(name="Unique_Target_Title_Count")
    .sort_values("Unique_Target_Title_Count", ascending=False)
)

source_type_summary = (
    final["Source_Type"]
    .value_counts(dropna=False)
    .reset_index()
)

source_type_summary.columns = ["Source_Type", "Row_Count"]

source_ai_summary = (
    final["Source_AI_Exposure_Group"]
    .value_counts(dropna=False)
    .reset_index()
)

source_ai_summary.columns = ["Source_AI_Exposure_Group", "Row_Count"]

target_ai_summary = (
    final["Target_AI_Exposure_Group"]
    .value_counts(dropna=False)
    .reset_index()
)

target_ai_summary.columns = ["Target_AI_Exposure_Group", "Row_Count"]

print("\nFinal rows:", final.shape)
print("Gateway pathway rows:", gateway_paths.shape)
print("Duplicate Source_SOC + Target_SOC rows:", len(duplicate_check))

print("\nSource AI summary:")
print(source_ai_summary)

print("\nSource type summary:")
print(source_type_summary)

print("\nTarget AI summary:")
print(target_ai_summary)

# ============================================================
# REORDER FINAL COLUMNS
# ============================================================

first_cols = [
    "Source_SOC",
    "Source_Occupation",
    "Source_AI_Exposure_Group",
    "Compat_Rank",
    "Target_SOC",
    "Target_Occupation",
    "Target_AI_Exposure_Group",
    "Source_Type",
    "ONET_Tier",
    "ONET_Index",
    "Final_Compat_Index",
    "Weighted_Compat_Index",
    "Avg_Compat_Index",
    "Min_Compat_Index",
    "Max_Compat_Index",
    "Median_Hourly_Earnings",
    "Jobs_2020",
    "Jobs_2025",
    "Change_2020_2025",
    "Estimated_Annual_Openings",
    "Source_Row_Count",
    "Lightcast_Median_Hourly_Earnings",
    "Lightcast_Median_Annual_Earnings",
    "Lightcast_2025_Jobs",
    "Lightcast_2024_2025_Change",
    "Lightcast_2024_2025_Estimated_Annual_Openings",
    "Lightcast_Top_Education_Level",
    "Lightcast_Top_Education_Percent",
    "AI_Composite_Annual_Total_Openings",
    "AI_Composite_Annual_Percent_Change",
    "AI_Composite_Education_Value",
    "AI_Composite_Work_Experience_Value",
    "AI_Composite_Growth_Rate",
    "AI_Composite_50th_Percentile_Wage",
    "Target_SOC_Detailed_Options",
    "Target_Occupation_Options",
    "Source_Occupation_Original",
    "Target_Occupation_Original",
    "Base_Title",
    "Related_Title"
]

existing_first_cols = [c for c in first_cols if c in final.columns]
remaining_cols = [c for c in final.columns if c not in existing_first_cols]

final = final[existing_first_cols + remaining_cols]

# ============================================================
# EXPORT
# ============================================================

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    final.to_excel(writer, sheet_name="Final_Transitions", index=False)
    gateway_paths.to_excel(writer, sheet_name="Gateway_Pathways", index=False)

    merged.to_excel(writer, sheet_name="Lightcast_With_ONET_Match", index=False)
    onet_only_formatted.to_excel(writer, sheet_name="ONET_Only_Appended", index=False)

    lightcast_economic.to_excel(writer, sheet_name="Lightcast_Economic_Data", index=False)
    lightcast_education.to_excel(writer, sheet_name="Lightcast_Education", index=False)
    lightcast_top_education.to_excel(writer, sheet_name="Lightcast_Top_Education", index=False)
    lightcast_ksas.to_excel(writer, sheet_name="Lightcast_KSAs", index=False)
    lightcast_errors.to_excel(writer, sheet_name="Lightcast_File_Errors", index=False)

    ai_composite_economic.to_excel(writer, sheet_name="AI_Composite_Economic", index=False)

    soc_title_lookup.to_excel(writer, sheet_name="SOC_Title_Lookup", index=False)
    ai_lookup.to_excel(writer, sheet_name="AI_Exposure_Lookup", index=False)
    source_ai_summary.to_excel(writer, sheet_name="Source_AI_Summary", index=False)
    source_type_summary.to_excel(writer, sheet_name="Source_Type_Summary", index=False)
    target_ai_summary.to_excel(writer, sheet_name="Target_AI_Summary", index=False)
    duplicate_check.to_excel(writer, sheet_name="Duplicate_Check", index=False)
    source_title_check.to_excel(writer, sheet_name="Source_Title_Check", index=False)
    target_title_check.to_excel(writer, sheet_name="Target_Title_Check", index=False)

print("\nSaved final file:")
print(OUTPUT_FILE)

print("\nDone.")

Loaded crosswalk rows: (1016, 7)
Using SOC column: SOC
Using title column: Occupation Title
Using exposure column: AI Exposure Category
High + Very High SOCs from composite file: 303
Using Lightcast sheet: Sheet1
Lightcast rows after High + Very High source filter: (1288, 22)
O*NET rows after High + Very High source filter and collapse: (6213, 8)
O*NET-only rows to append: (5476, 8)

Final rows: (6764, 46)
Gateway pathway rows: (6820, 20)
Duplicate Source_SOC + Target_SOC rows: 0

Source AI summary:
  Source_AI_Exposure_Group  Row_Count
0                Very High       3690
1                     High       3074

Source type summary:
  Source_Type  Row_Count
0        ONET       5476
1        Both        737
2   Lightcast        551

Target AI summary:
  Target_AI_Exposure_Group  Row_Count
0                Very High       2890
1                     High       1923
2                   Medium       1287
3                      Low        334
4                      NaN        227
5          

In [6]:
print("\nColumns after AI exposure merges:")
for c in final.columns:
    if "Exposure" in c or "AI" in c:
        print(c)

print("\nFirst 10 rows of exposure-related columns:")

exposure_cols = [
    c for c in final.columns
    if "Exposure" in c or "AI" in c
]

print(final[exposure_cols].head(10))


Columns after AI exposure merges:
Source_AI_Exposure_Group_x
Source_AI_Exposure_Group_y
Target_AI_Exposure_Group

First 10 rows of exposure-related columns:
  Source_AI_Exposure_Group_x Source_AI_Exposure_Group_y  \
0                  Very High                  Very High   
1                  Very High                  Very High   
2                  Very High                  Very High   
3                  Very High                  Very High   
4                  Very High                  Very High   
5                  Very High                  Very High   
6                  Very High                  Very High   
7                  Very High                  Very High   
8                  Very High                  Very High   
9                  Very High                  Very High   

  Target_AI_Exposure_Group  
0                Very High  
1                     High  
2                Very High  
3                     High  
4                Very High  
5                 

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from datetime import datetime

# ============================================================
# PATHS
# ============================================================

FINAL_FILE = Path(
    r"C:\Users\301533\Documents\AI Analysis\Ligthcast Skills & RO\FINAL_High_AND_VeryHigh_AI_Lightcast_ONET_Transitions_WITH_GATEWAYS.xlsx"
)

AI_FILE = Path(
    r"G:\Shared drives\EO_ECONOMIC ANALYSIS\ea_common\Projects\AI\risk_index\ai_exposure_composite_clean.csv"
)

ONET_FILE = Path(
    r"C:\Users\301533\Documents\Green Analysis\Related Occupations.xlsx"
)

OUT_FILE = FINAL_FILE.with_name(
    f"QA_Check_Final_Transitions_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
)

# ============================================================
# HELPERS
# ============================================================

month_map = {
    "Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04",
    "May": "05", "Jun": "06", "Jul": "07", "Aug": "08",
    "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"
}

def fix_soc(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().replace(".0", "")

    if s.lower() in ["nan", "none", ""]:
        return np.nan

    match = re.search(r"(\d{2})-(\d{4})", s)
    if match:
        return f"{match.group(1)}-{match.group(2)}"

    if "-" in s:
        left, right = s.split("-", 1)

        if left in month_map and right.isdigit():
            if len(right) <= 2:
                return f"{month_map[left]}-20{right.zfill(2)}"
            return f"{month_map[left]}-{right.zfill(4)}"

        if right in month_map and left.isdigit():
            return f"{month_map[right]}-20{left.zfill(2)}"

    return s

def is_valid_soc(x):
    if pd.isna(x):
        return False
    return bool(re.match(r"^\d{2}-\d{4}$", str(x).strip()))

def clean_num(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in ["", "-", "NA", "N/A", "nan", "None"]:
        return np.nan
    return pd.to_numeric(
        s.replace("$", "").replace(",", "").replace("%", "").replace("−", "-"),
        errors="coerce"
    )

# ============================================================
# LOAD FINAL WORKBOOK
# ============================================================

xls = pd.ExcelFile(FINAL_FILE)
print("Sheets:")
print(xls.sheet_names)

final = pd.read_excel(FINAL_FILE, sheet_name="Final_Transitions")
final.columns = final.columns.astype(str).str.strip()

print("\nFinal_Transitions shape:", final.shape)
print("\nColumns:")
print(final.columns.tolist())

# ============================================================
# SOC CODE QA
# ============================================================

qa = final.copy()

qa["Source_SOC_Original"] = qa["Source_SOC"]
qa["Target_SOC_Original"] = qa["Target_SOC"]

qa["Source_SOC_Fixed"] = qa["Source_SOC"].apply(fix_soc)
qa["Target_SOC_Fixed"] = qa["Target_SOC"].apply(fix_soc)

qa["Source_SOC_Changed"] = qa["Source_SOC_Original"].astype(str) != qa["Source_SOC_Fixed"].astype(str)
qa["Target_SOC_Changed"] = qa["Target_SOC_Original"].astype(str) != qa["Target_SOC_Fixed"].astype(str)

qa["Source_SOC_Valid"] = qa["Source_SOC_Fixed"].apply(is_valid_soc)
qa["Target_SOC_Valid"] = qa["Target_SOC_Fixed"].apply(is_valid_soc)

bad_source_socs = qa[~qa["Source_SOC_Valid"]].copy()
bad_target_socs = qa[~qa["Target_SOC_Valid"]].copy()

changed_socs = qa[
    qa["Source_SOC_Changed"] | qa["Target_SOC_Changed"]
].copy()

print("\nBad source SOC rows:", len(bad_source_socs))
print("Bad target SOC rows:", len(bad_target_socs))
print("Changed SOC rows after fix:", len(changed_socs))

# ============================================================
# DUPLICATE / RELATIONSHIP QA
# ============================================================

duplicate_pairs = qa[
    qa.duplicated(subset=["Source_SOC_Fixed", "Target_SOC_Fixed"], keep=False)
].sort_values(["Source_SOC_Fixed", "Target_SOC_Fixed"])

source_summary = (
    qa.groupby(["Source_SOC_Fixed", "Source_Occupation", "Source_AI_Exposure_Group"], dropna=False)
    .agg(
        Related_Occupation_Count=("Target_SOC_Fixed", "nunique"),
        Row_Count=("Target_SOC_Fixed", "size")
    )
    .reset_index()
    .sort_values("Related_Occupation_Count", ascending=False)
)

target_exposure_summary = (
    qa["Target_AI_Exposure_Group"]
    .value_counts(dropna=False)
    .reset_index()
)
target_exposure_summary.columns = ["Target_AI_Exposure_Group", "Row_Count"]

source_type_summary = (
    qa["Source_Type"]
    .value_counts(dropna=False)
    .reset_index()
)
source_type_summary.columns = ["Source_Type", "Row_Count"]

# ============================================================
# LIGHTCAST SCORE QA
# ============================================================

score_cols = [
    "Final_Compat_Index",
    "Weighted_Compat_Index",
    "Avg_Compat_Index",
    "Min_Compat_Index",
    "Max_Compat_Index",
    "ONET_Index"
]

for col in score_cols:
    if col in qa.columns:
        qa[col] = qa[col].apply(clean_num)

score_summary = qa[score_cols].describe(include="all").reset_index()

lightcast_score_check = qa[
    qa["Source_Type"].isin(["Lightcast", "Both"])
].copy()

lightcast_score_check = lightcast_score_check[
    lightcast_score_check["Final_Compat_Index"].isna()
].copy()

onet_score_check = qa[
    qa["Source_Type"].isin(["ONET", "Both"])
].copy()

onet_score_check = onet_score_check[
    onet_score_check["ONET_Index"].isna() | onet_score_check["ONET_Tier"].isna()
].copy()

print("\nLightcast/Both missing compat score:", len(lightcast_score_check))
print("ONET/Both missing ONET tier/index:", len(onet_score_check))

# ============================================================
# COMPARE FINAL SOCS TO AI COMPOSITE
# ============================================================

ai = pd.read_csv(AI_FILE)
ai.columns = ai.columns.astype(str).str.strip()

ai_soc_col = "SOC" if "SOC" in ai.columns else None
if ai_soc_col is None:
    raise ValueError("AI file does not have SOC column. Check columns.")

ai["SOC_Fixed"] = ai[ai_soc_col].apply(fix_soc)

ai_lookup = ai[["SOC_Fixed"]].drop_duplicates()
ai_soc_set = set(ai_lookup["SOC_Fixed"].dropna())

source_not_in_ai = qa[~qa["Source_SOC_Fixed"].isin(ai_soc_set)].copy()
target_not_in_ai = qa[~qa["Target_SOC_Fixed"].isin(ai_soc_set)].copy()

print("\nSource SOCs not in AI composite:", source_not_in_ai["Source_SOC_Fixed"].nunique())
print("Target SOCs not in AI composite:", target_not_in_ai["Target_SOC_Fixed"].nunique())

# ============================================================
# EXPORT QA WORKBOOK
# ============================================================

with pd.ExcelWriter(OUT_FILE, engine="openpyxl") as writer:
    qa.to_excel(writer, sheet_name="QA_All_Rows", index=False)
    bad_source_socs.to_excel(writer, sheet_name="Bad_Source_SOC", index=False)
    bad_target_socs.to_excel(writer, sheet_name="Bad_Target_SOC", index=False)
    changed_socs.to_excel(writer, sheet_name="Changed_SOC_After_Fix", index=False)
    duplicate_pairs.to_excel(writer, sheet_name="Duplicate_Source_Target", index=False)
    source_summary.to_excel(writer, sheet_name="Source_Summary", index=False)
    source_type_summary.to_excel(writer, sheet_name="Source_Type_Summary", index=False)
    target_exposure_summary.to_excel(writer, sheet_name="Target_Exposure_Summary", index=False)
    score_summary.to_excel(writer, sheet_name="Score_Summary", index=False)
    lightcast_score_check.to_excel(writer, sheet_name="Missing_Lightcast_Score", index=False)
    onet_score_check.to_excel(writer, sheet_name="Missing_ONET_Info", index=False)
    source_not_in_ai.to_excel(writer, sheet_name="Source_Not_In_AI", index=False)
    target_not_in_ai.to_excel(writer, sheet_name="Target_Not_In_AI", index=False)

print("\nSaved QA workbook:")
print(OUT_FILE)

Sheets:
['Final_Transitions', 'Gateway_Pathways', 'Lightcast_With_ONET_Match', 'ONET_Only_Appended', 'Lightcast_Economic_Data', 'Lightcast_Education', 'Lightcast_Top_Education', 'Lightcast_KSAs', 'Lightcast_File_Errors', 'AI_Composite_Economic', 'SOC_Title_Lookup', 'AI_Exposure_Lookup', 'Source_AI_Summary', 'Source_Type_Summary', 'Target_AI_Summary', 'Duplicate_Check', 'Source_Title_Check', 'Target_Title_Check']

Final_Transitions shape: (6764, 46)

Columns:
['Source_SOC', 'Source_Occupation', 'Source_AI_Exposure_Group', 'Compat_Rank', 'Target_SOC', 'Target_Occupation', 'Target_AI_Exposure_Group', 'Source_Type', 'ONET_Tier', 'ONET_Index', 'Final_Compat_Index', 'Weighted_Compat_Index', 'Avg_Compat_Index', 'Min_Compat_Index', 'Max_Compat_Index', 'Median_Hourly_Earnings', 'Jobs_2020', 'Jobs_2025', 'Change_2020_2025', 'Estimated_Annual_Openings', 'Source_Row_Count', 'Lightcast_Median_Hourly_Earnings', 'Lightcast_Median_Annual_Earnings', 'Lightcast_2025_Jobs', 'Lightcast_2024_2025_Change', 

In [4]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from datetime import datetime

FINAL_FILE = Path(
    r"C:\Users\301533\Documents\AI Analysis\Ligthcast Skills & RO\FINAL_High_AND_VeryHigh_AI_Lightcast_ONET_Transitions_WITH_GATEWAYS.xlsx"
)

AI_FILE = Path(
    r"G:\Shared drives\EO_ECONOMIC ANALYSIS\ea_common\Projects\AI\risk_index\ai_exposure_composite_clean.csv"
)

OUT_FILE = FINAL_FILE.with_name(
    f"Final_Transitions_WITH_Normalized_Relatedness_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
)

def clean_num(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s in ["", "-", "NA", "N/A", "nan", "None"]:
        return np.nan
    return pd.to_numeric(
        s.replace("$", "").replace(",", "").replace("%", "").replace("−", "-"),
        errors="coerce"
    )

def fix_soc(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(".0", "")
    match = re.search(r"(\d{2})-(\d{4})", s)
    if match:
        return f"{match.group(1)}-{match.group(2)}"
    return s

final = pd.read_excel(FINAL_FILE, sheet_name="Final_Transitions")
final.columns = final.columns.astype(str).str.strip()

final["Source_SOC"] = final["Source_SOC"].apply(fix_soc)
final["Target_SOC"] = final["Target_SOC"].apply(fix_soc)

ai = pd.read_csv(AI_FILE)
ai.columns = ai.columns.astype(str).str.strip()
ai["SOC"] = ai["SOC"].apply(fix_soc)

ai_socs = set(ai["SOC"].dropna())

target_not_in_ai = (
    final.loc[~final["Target_SOC"].isin(ai_socs), [
        "Target_SOC",
        "Target_Occupation",
        "Target_AI_Exposure_Group",
        "Source_SOC",
        "Source_Occupation",
        "Source_Type",
        "ONET_Tier",
        "Final_Compat_Index"
    ]]
    .drop_duplicates()
    .sort_values(["Target_SOC", "Source_SOC"])
)

# ============================================================
# NORMALIZE LIGHTCAST + ONET USING BOTH SCORES AND RANKS
# ============================================================

final["Final_Compat_Index"] = final["Final_Compat_Index"].apply(clean_num)
final["ONET_Index"] = final["ONET_Index"].apply(clean_num)

# 1 = best rank
final["Lightcast_Rank"] = (
    final.groupby("Source_SOC")["Final_Compat_Index"]
    .rank(method="first", ascending=False)
)

final["ONET_Rank"] = (
    final.groupby("Source_SOC")["ONET_Index"]
    .rank(method="first", ascending=True)
)

# Convert ranks to 0 to 1 scores, where 1 = strongest
final["Lightcast_Rank_Score"] = (
    1 - (
        (final["Lightcast_Rank"] - 1) /
        (final.groupby("Source_SOC")["Lightcast_Rank"].transform("max") - 1)
    )
)

final["ONET_Rank_Score"] = (
    1 - (
        (final["ONET_Rank"] - 1) /
        (final.groupby("Source_SOC")["ONET_Rank"].transform("max") - 1)
    )
)

# If a source has only one related occupation, give it a full rank score
final["Lightcast_Rank_Score"] = final["Lightcast_Rank_Score"].fillna(1)
final["ONET_Rank_Score"] = final["ONET_Rank_Score"].fillna(1)

# O*NET tier strength
final["ONET_Tier_Score"] = final["ONET_Tier"].map({
    "Primary-Short": 1.00,
    "Primary-Long": 0.75,
    "Supplemental": 0.50
})

# Lightcast compatibility score, assuming 0 to 100
final["Lightcast_Compat_Score"] = final["Final_Compat_Index"] / 100

already_decimal = final["Final_Compat_Index"].between(0, 1, inclusive="both")
final.loc[already_decimal, "Lightcast_Compat_Score"] = final.loc[
    already_decimal, "Final_Compat_Index"
]

# Lightcast combined source score: score + rank
final["Lightcast_Combined_Score"] = (
    0.70 * final["Lightcast_Compat_Score"] +
    0.30 * final["Lightcast_Rank_Score"]
)

# O*NET combined source score: tier + rank
final["ONET_Combined_Score"] = (
    0.70 * final["ONET_Tier_Score"] +
    0.30 * final["ONET_Rank_Score"]
)

# Shared combined score
def combined_score(row):
    lc = row["Lightcast_Combined_Score"]
    onet = row["ONET_Combined_Score"]
    source_type = str(row["Source_Type"]).strip()

    if source_type == "Both" and pd.notna(lc) and pd.notna(onet):
        return min(((lc + onet) / 2) + 0.10, 1.00)

    if source_type == "Lightcast" and pd.notna(lc):
        return lc

    if source_type == "ONET" and pd.notna(onet):
        return onet

    if pd.notna(lc) and pd.notna(onet):
        return (lc + onet) / 2

    if pd.notna(lc):
        return lc

    if pd.notna(onet):
        return onet

    return np.nan

final["Both_Combined_Score"] = final.apply(combined_score, axis=1)

final["Both_Combined_Score_100"] = (
    final["Both_Combined_Score"] * 100
).round(1)

final["Both_Combined_Rank"] = (
    final.groupby("Source_SOC")["Both_Combined_Score"]
    .rank(method="first", ascending=False)
)

final = final.sort_values(
    ["Source_SOC", "Both_Combined_Rank", "Target_SOC"]
).copy()

score_summary = (
    final.groupby("Source_Type", dropna=False)
    .agg(
        Rows=("Target_SOC", "size"),
        Avg_Lightcast_Compat_Score=("Lightcast_Compat_Score", "mean"),
        Avg_Lightcast_Rank_Score=("Lightcast_Rank_Score", "mean"),
        Avg_Lightcast_Combined_Score=("Lightcast_Combined_Score", "mean"),
        Avg_ONET_Tier_Score=("ONET_Tier_Score", "mean"),
        Avg_ONET_Rank_Score=("ONET_Rank_Score", "mean"),
        Avg_ONET_Combined_Score=("ONET_Combined_Score", "mean"),
        Avg_Both_Combined_Score=("Both_Combined_Score", "mean"),
        Min_Both_Combined_Score=("Both_Combined_Score", "min"),
        Max_Both_Combined_Score=("Both_Combined_Score", "max")
    )
    .reset_index()
)

with pd.ExcelWriter(OUT_FILE, engine="openpyxl") as writer:
    final.to_excel(writer, sheet_name="Final_Transitions_Scored", index=False)
    target_not_in_ai.to_excel(writer, sheet_name="Target_SOC_Not_In_AI", index=False)
    score_summary.to_excel(writer, sheet_name="Score_Summary", index=False)

print("Saved scored review file:")
print(OUT_FILE)

print("\nTarget SOCs not in AI composite:", target_not_in_ai["Target_SOC"].nunique())
print("\nScore summary:")
print(score_summary)

Saved scored review file:
C:\Users\301533\Documents\AI Analysis\Ligthcast Skills & RO\Final_Transitions_WITH_Normalized_Relatedness_20260512_215718.xlsx

Target SOCs not in AI composite: 24

Score summary:
  Source_Type  Rows  Avg_Lightcast_Compat_Score  Avg_Lightcast_Rank_Score  \
0        Both   737                    0.938081                  0.616287   
1   Lightcast   551                    0.931255                  0.344458   
2        ONET  5476                         NaN                  1.000000   

   Avg_Lightcast_Combined_Score  Avg_ONET_Tier_Score  Avg_ONET_Rank_Score  \
0                      0.841543             0.792741             0.634038   
1                      0.755216                  NaN             1.000000   
2                           NaN             0.681702             0.481960   

   Avg_ONET_Combined_Score  Avg_Both_Combined_Score  Min_Both_Combined_Score  \
0                 0.745130                 0.880477                 0.601545   
1               

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from datetime import datetime

# ============================================================
# PATHS
# ============================================================

LIGHTCAST_HIGH_FOLDER = Path(
    r"C:\Users\301533\Documents\AI Analysis\Ligthcast Skills & RO\High AI"
)

LIGHTCAST_VERY_HIGH_FOLDER = Path(
    r"C:\Users\301533\Documents\AI Analysis\Ligthcast Skills & RO"
)

ONET_FILE = Path(
    r"C:\Users\301533\Downloads\Related Occupations (2).xlsx"
)

CROSSWALK_FILE = Path(
    r"C:\Users\301533\Documents\AI Analysis\2019_to_SOC_Crosswalk.xlsx"
)

AI_FILE = Path(
    r"G:\Shared drives\EO_ECONOMIC ANALYSIS\ea_common\Projects\AI\risk_index\ai_exposure_composite_clean.csv"
)

OUT_FILE = Path(
    rf"C:\Users\301533\Documents\AI Analysis\Ligthcast Skills & RO\FINAL_REBUILT_High_VeryHigh_AND_Medium_Related_Occs_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
)

SOURCE_EXPOSURE_GROUPS = ["Medium", "High", "Very High"]

HIGH_VERY_HIGH_TARGET_GROUPS = ["Very Low", "Low", "Medium"]
MEDIUM_TARGET_GROUPS = ["Very Low", "Low"]

# ============================================================
# HELPERS
# ============================================================

month_map = {
    "Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04",
    "May": "05", "Jun": "06", "Jul": "07", "Aug": "08",
    "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"
}

def clean_num(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    if s in ["", "-", "NA", "N/A", "nan", "None"]:
        return np.nan

    neg = False
    if s.startswith("(") and s.endswith(")"):
        neg = True
        s = s.replace("(", "").replace(")", "")

    s = (
        s.replace("$", "")
         .replace("/hr", "")
         .replace(",", "")
         .replace("%", "")
         .replace("−", "-")
         .strip()
    )

    out = pd.to_numeric(s, errors="coerce")

    if pd.notna(out) and neg:
        out = -out

    return out

def clean_title(x):
    if pd.isna(x):
        return np.nan

    return (
        str(x)
        .strip()
        .lower()
        .replace("*", "")
        .replace(",", "")
        .replace("  ", " ")
    )

def fix_soc_basic(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().replace(".0", "")

    if s.lower() in ["nan", "none", ""]:
        return np.nan

    match = re.search(r"(\d{2})-(\d{4})", s)
    if match:
        return f"{match.group(1)}-{match.group(2)}"

    if "-" in s:
        left, right = s.split("-", 1)

        if left in month_map and right.isdigit():
            if len(right) <= 2:
                return f"{month_map[left]}-20{right.zfill(2)}"
            return f"{month_map[left]}-{right.zfill(4)}"

        if right in month_map and left.isdigit():
            return f"{month_map[right]}-20{left.zfill(2)}"

    return s

def fix_onet_detailed(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    match = re.search(r"(\d{2}-\d{4}(?:\.\d{2})?)", s)
    if match:
        return match.group(1)

    return fix_soc_basic(s)

def extract_occ_and_onet(value):
    if pd.isna(value):
        return np.nan, np.nan

    s = str(value).strip()

    match = re.search(r"^(.*?)\s*\((\d{2}-\d{4}(?:\.\d{2})?)\)", s)

    if match:
        return match.group(1).strip(), match.group(2).strip()

    return s, np.nan

def get_parameters_source(file):
    params = pd.read_excel(file, sheet_name="Parameters", header=None)

    source_onet = np.nan
    source_title = np.nan

    for i in range(len(params)):
        vals = params.iloc[i].astype(str).str.strip().tolist()

        if len(vals) >= 2 and vals[0] == "Code" and vals[1] == "Description":
            if i + 1 < len(params):
                source_onet = params.iloc[i + 1, 0]
                source_title = params.iloc[i + 1, 1]
                break

    return fix_onet_detailed(source_onet), source_title

# ============================================================
# LOAD CROSSWALK
# ============================================================

crosswalk = pd.read_excel(CROSSWALK_FILE, header=3)
crosswalk.columns = crosswalk.columns.astype(str).str.strip()

crosswalk = crosswalk.rename(columns={
    "O*NET-SOC 2019 Code": "ONET_2019_Code",
    "O*NET-SOC 2019 Title": "ONET_2019_Title",
    "2018 SOC Code": "SOC_2018",
    "2018 SOC Title": "SOC_2018_Title"
})

crosswalk["ONET_2019_Code_Clean"] = crosswalk["ONET_2019_Code"].apply(fix_onet_detailed)
crosswalk["SOC_2018_Clean"] = crosswalk["SOC_2018"].apply(fix_soc_basic)
crosswalk["ONET_Title_Clean"] = crosswalk["ONET_2019_Title"].apply(clean_title)

onet_to_soc2018 = (
    crosswalk
    .dropna(subset=["ONET_2019_Code_Clean", "SOC_2018_Clean"])
    .drop_duplicates(subset=["ONET_2019_Code_Clean"])
    .set_index("ONET_2019_Code_Clean")["SOC_2018_Clean"]
    .to_dict()
)

title_to_soc2018 = (
    crosswalk
    .dropna(subset=["ONET_Title_Clean", "SOC_2018_Clean"])
    .drop_duplicates(subset=["ONET_Title_Clean"])
    .set_index("ONET_Title_Clean")["SOC_2018_Clean"]
    .to_dict()
)

soc_title_lookup = (
    crosswalk[["SOC_2018_Clean", "SOC_2018_Title"]]
    .dropna()
    .drop_duplicates(subset=["SOC_2018_Clean"])
    .copy()
)

soc_title_dict = dict(
    zip(soc_title_lookup["SOC_2018_Clean"], soc_title_lookup["SOC_2018_Title"])
)

def onet_to_soc(onet_code, title=None):
    onet_clean = fix_onet_detailed(onet_code)

    if pd.notna(onet_clean) and onet_clean in onet_to_soc2018:
        return onet_to_soc2018[onet_clean]

    title_clean = clean_title(title)
    if pd.notna(title_clean) and title_clean in title_to_soc2018:
        return title_to_soc2018[title_clean]

    return fix_soc_basic(onet_clean)

print("Crosswalk rows:", len(crosswalk))

# ============================================================
# LOAD AI COMPOSITE
# ============================================================

ai = pd.read_csv(AI_FILE)
ai.columns = ai.columns.astype(str).str.strip()

possible_soc_cols = [
    "SOC", "SOC Code", "SOC_Code", "SOC2018", "SOC2018.1",
    "O*NET-SOC Code", "O*NET_SOC_Code", "O*NET-SOC Code.1"
]

possible_title_cols = [
    "Occupation Title", "Title", "Occupation", "2018 SOC Title", "O*NET-SOC 2019 Title"
]

possible_exposure_cols = [
    "AI Exposure Category", "AI_Exposure_Group", "exposure", "Exposure",
    "AI Exposure Group", "AI_Exposure_Category", "Exposure_Group"
]

soc_col = next((c for c in possible_soc_cols if c in ai.columns), None)
title_col = next((c for c in possible_title_cols if c in ai.columns), None)
exposure_col = next((c for c in possible_exposure_cols if c in ai.columns), None)

if soc_col is None:
    raise ValueError("Could not find SOC column in AI composite.")

if title_col is None:
    raise ValueError("Could not find title column in AI composite.")

if exposure_col is None:
    raise ValueError("Could not find exposure column in AI composite.")

ai["Title_Clean"] = ai[title_col].apply(clean_title)
ai["SOC_From_Title"] = ai["Title_Clean"].map(title_to_soc2018)
ai["SOC_Fallback"] = ai[soc_col].apply(fix_soc_basic)
ai["SOC"] = ai["SOC_From_Title"].fillna(ai["SOC_Fallback"])
ai["AI_Exposure_Group"] = ai[exposure_col].astype(str).str.strip()

ai_lookup = (
    ai[["SOC", "AI_Exposure_Group"]]
    .dropna()
    .drop_duplicates(subset=["SOC"])
    .copy()
)

source_socs = set(
    ai_lookup.loc[
        ai_lookup["AI_Exposure_Group"].isin(SOURCE_EXPOSURE_GROUPS),
        "SOC"
    ]
)

ai_econ_cols = [
    "SOC",
    title_col,
    "AI_Exposure_Group",
    "Annual Total Openings",
    "Annual Percent Change",
    "Education Value",
    "Work Experience Value",
    "Growth Rate",
    "50th Percentile Wage"
]

ai_econ_cols = [c for c in ai_econ_cols if c in ai.columns]

ai_economic = ai[ai_econ_cols].copy()

ai_economic = ai_economic.rename(columns={
    title_col: "AI_Occupation_Title",
    "Annual Total Openings": "AI_Annual_Total_Openings",
    "Annual Percent Change": "AI_Annual_Percent_Change",
    "Education Value": "AI_Education_Value",
    "Work Experience Value": "AI_Work_Experience_Value",
    "Growth Rate": "AI_Growth_Rate",
    "50th Percentile Wage": "AI_50th_Percentile_Wage"
})

ai_economic = ai_economic.drop_duplicates(subset=["SOC"])

print("Medium + High + Very High source SOCs:", len(source_socs))

# ============================================================
# PARSE LIGHTCAST INDIVIDUAL FILES
# ============================================================

lightcast_files = []
lightcast_files += list(LIGHTCAST_HIGH_FOLDER.glob("Skills_Transferability_*.xlsx"))
lightcast_files += list(LIGHTCAST_VERY_HIGH_FOLDER.glob("Skills_Transferability_*.xlsx"))
lightcast_files = sorted(set(lightcast_files))

lc_rows = []
lc_source_econ_rows = []
lc_source_edu_rows = []
lc_errors = []

for file in lightcast_files:
    try:
        xls = pd.ExcelFile(file)

        if "Parameters" not in xls.sheet_names:
            continue

        source_onet, source_title = get_parameters_source(file)
        source_soc = onet_to_soc(source_onet, source_title)

        if pd.isna(source_soc):
            lc_errors.append({
                "File": file.name,
                "Error": "Could not identify source SOC from Parameters"
            })
            continue

        if "Overview" in xls.sheet_names:
            overview = pd.read_excel(file, sheet_name="Overview", header=None)
            overview_metrics = {}

            for _, row in overview.iterrows():
                label = row.iloc[0] if len(row) > 0 else np.nan
                value = row.iloc[1] if len(row) > 1 else np.nan

                if isinstance(label, str):
                    label_clean = label.strip()
                    overview_metrics[label_clean] = value

            source_median_hourly = clean_num(overview_metrics.get("Median Hourly Earnings*"))

            lc_source_econ_rows.append({
                "Source_SOC": source_soc,
                "Source_ONET_SOC": source_onet,
                "Source_Occupation": source_title,
                "LC_Source_Median_Hourly_Earnings": source_median_hourly,
                "LC_Source_Median_Annual_Earnings": source_median_hourly * 2080 if pd.notna(source_median_hourly) else np.nan,
                "LC_Source_2025_Jobs": clean_num(overview_metrics.get("2025 Jobs*")),
                "LC_Source_2024_2025_Change": clean_num(overview_metrics.get("2024-2025 Change*")),
                "LC_Source_2024_2025_Estimated_Annual_Openings": clean_num(overview_metrics.get("2024-2025 Estimated Annual Openings*")),
                "Source_File": file.name
            })

        if "Edu Attainment" in xls.sheet_names:
            edu = pd.read_excel(file, sheet_name="Edu Attainment", header=None)
            edu = edu.dropna(how="all")

            for _, row in edu.iterrows():
                edu_level = row.iloc[0] if len(row) > 0 else np.nan
                edu_pct = row.iloc[1] if len(row) > 1 else np.nan

                if isinstance(edu_level, str):
                    edu_level = edu_level.strip()

                    if edu_level not in ["Educational Attainment", ""]:
                        pct = clean_num(edu_pct)

                        lc_source_edu_rows.append({
                            "Source_SOC": source_soc,
                            "Source_ONET_SOC": source_onet,
                            "Source_Occupation": source_title,
                            "LC_Source_Education_Level": edu_level,
                            "LC_Source_Education_Percent": pct / 100 if pd.notna(pct) and pct > 1 else pct,
                            "Source_File": file.name
                        })

        compatible_sheet = None

        for s in xls.sheet_names:
            if "top compatible" in str(s).lower().strip():
                compatible_sheet = s
                break

        if compatible_sheet is None:
            lc_errors.append({
                "File": file.name,
                "Source_SOC": source_soc,
                "Error": "Missing Top Compatible Occs sheet",
                "Sheets": " | ".join(xls.sheet_names)
            })
            continue

        comp = pd.read_excel(
            file,
            sheet_name=compatible_sheet,
            header=2
        )

        comp.columns = comp.columns.astype(str).str.strip()
        comp = comp.dropna(how="all")

        comp = comp.rename(columns={
            "O*NET Occupation": "Target_ONET_Occupation_Raw",
            "Median Hourly Earnings*": "LC_Target_Median_Hourly_Earnings",
            "2024 Jobs*": "LC_Target_2024_Jobs",
            "2025 Jobs*": "LC_Target_2025_Jobs",
            "2024-2025 Change*": "LC_Target_2024_2025_Change",
            "2024-2025 Estimated Annual Openings*": "LC_Target_2024_2025_Estimated_Annual_Openings",
            "Compat. Index": "Lightcast_Compat_Index"
        })

        if "Target_ONET_Occupation_Raw" not in comp.columns:
            lc_errors.append({
                "File": file.name,
                "Source_SOC": source_soc,
                "Sheet": compatible_sheet,
                "Error": "Missing target occupation column",
                "Columns": " | ".join(comp.columns)
            })
            continue

        for _, row in comp.iterrows():
            target_title, target_onet = extract_occ_and_onet(
                row.get("Target_ONET_Occupation_Raw")
            )

            target_soc = onet_to_soc(target_onet, target_title)

            if pd.isna(target_soc):
                continue

            if source_soc == target_soc:
                continue

            median_hourly = clean_num(row.get("LC_Target_Median_Hourly_Earnings"))

            lc_rows.append({
                "Source_SOC": source_soc,
                "Source_ONET_SOC": source_onet,
                "Source_Occupation": source_title,
                "Target_SOC": target_soc,
                "Target_ONET_SOC": target_onet,
                "Target_Occupation_Lightcast": target_title,
                "Final_Compat_Index": clean_num(row.get("Lightcast_Compat_Index")),
                "LC_Target_Median_Hourly_Earnings": median_hourly,
                "LC_Target_Median_Annual_Earnings": median_hourly * 2080 if pd.notna(median_hourly) else np.nan,
                "LC_Target_2024_Jobs": clean_num(row.get("LC_Target_2024_Jobs")),
                "LC_Target_2025_Jobs": clean_num(row.get("LC_Target_2025_Jobs")),
                "LC_Target_2024_2025_Change": clean_num(row.get("LC_Target_2024_2025_Change")),
                "LC_Target_2024_2025_Estimated_Annual_Openings": clean_num(row.get("LC_Target_2024_2025_Estimated_Annual_Openings")),
                "Lightcast_Source_File": file.name
            })

    except Exception as e:
        lc_errors.append({
            "File": file.name,
            "Error": str(e)
        })

lc_raw = pd.DataFrame(lc_rows)
lc_source_econ = pd.DataFrame(lc_source_econ_rows)
lc_source_edu = pd.DataFrame(lc_source_edu_rows)
lc_errors = pd.DataFrame(lc_errors)

if lc_raw.empty:
    raise ValueError("No Lightcast compatible occupation rows were parsed.")

lc_raw["Source_SOC"] = lc_raw["Source_SOC"].apply(fix_soc_basic)
lc_raw["Target_SOC"] = lc_raw["Target_SOC"].apply(fix_soc_basic)

lc_raw = lc_raw[lc_raw["Source_SOC"].isin(source_socs)].copy()

lc = (
    lc_raw
    .sort_values(["Source_SOC", "Target_SOC", "Final_Compat_Index"], ascending=[True, True, False])
    .drop_duplicates(subset=["Source_SOC", "Target_SOC"], keep="first")
    .copy()
)

lc["Lightcast_Rank"] = (
    lc.groupby("Source_SOC")["Final_Compat_Index"]
    .rank(method="first", ascending=False)
)

print("Parsed Lightcast rows:", len(lc))
print("Lightcast source SOCs:", lc["Source_SOC"].nunique())

# ============================================================
# LOAD ONET RELATED OCCUPATIONS
# ============================================================

onet = pd.read_excel(ONET_FILE)
onet.columns = onet.columns.astype(str).str.strip()

onet = onet.rename(columns={
    "O*NET-SOC Code": "Source_ONET_SOC",
    "Title": "Source_Occupation_ONET",
    "Related O*NET-SOC Code": "Target_ONET_SOC",
    "Related Title": "Target_Occupation_ONET",
    "Relatedness Tier": "ONET_Tier",
    "Index": "ONET_Index"
})

onet["Source_ONET_SOC"] = onet["Source_ONET_SOC"].apply(fix_onet_detailed)
onet["Target_ONET_SOC"] = onet["Target_ONET_SOC"].apply(fix_onet_detailed)

onet["Source_SOC"] = onet.apply(
    lambda r: onet_to_soc(r["Source_ONET_SOC"], r["Source_Occupation_ONET"]),
    axis=1
)

onet["Target_SOC"] = onet.apply(
    lambda r: onet_to_soc(r["Target_ONET_SOC"], r["Target_Occupation_ONET"]),
    axis=1
)

onet = onet.dropna(subset=["Source_SOC", "Target_SOC"]).copy()
onet = onet[onet["Source_SOC"].isin(source_socs)].copy()
onet = onet[onet["Source_SOC"] != onet["Target_SOC"]].copy()

tier_priority = {
    "Primary-Short": 1,
    "Primary-Long": 2,
    "Supplemental": 3
}

onet["ONET_Tier_Priority"] = onet["ONET_Tier"].map(tier_priority)
onet["ONET_Index"] = onet["ONET_Index"].apply(clean_num)

onet = (
    onet
    .sort_values(["Source_SOC", "Target_SOC", "ONET_Tier_Priority", "ONET_Index"])
    .drop_duplicates(subset=["Source_SOC", "Target_SOC"], keep="first")
    .copy()
)

onet["ONET_Rank"] = (
    onet.groupby("Source_SOC")["ONET_Index"]
    .rank(method="first", ascending=True)
)

print("O*NET rows:", len(onet))
print("O*NET source SOCs:", onet["Source_SOC"].nunique())

# ============================================================
# MERGE LIGHTCAST + ONET
# ============================================================

merged = lc.merge(
    onet[
        [
            "Source_SOC",
            "Target_SOC",
            "Source_ONET_SOC",
            "Target_ONET_SOC",
            "Source_Occupation_ONET",
            "Target_Occupation_ONET",
            "ONET_Tier",
            "ONET_Index",
            "ONET_Rank"
        ]
    ],
    on=["Source_SOC", "Target_SOC"],
    how="outer",
    suffixes=("_Lightcast", "_ONET")
)

merged["Source_Type"] = np.select(
    [
        merged["Final_Compat_Index"].notna() & merged["ONET_Tier"].notna(),
        merged["Final_Compat_Index"].notna(),
        merged["ONET_Tier"].notna()
    ],
    [
        "Both",
        "Lightcast",
        "ONET"
    ],
    default="Unknown"
)

# ============================================================
# TITLES, EXPOSURE, ECONOMIC DATA
# ============================================================

merged["Source_Occupation"] = (
    merged["Source_SOC"].map(soc_title_dict)
    .fillna(merged.get("Source_Occupation"))
    .fillna(merged.get("Source_Occupation_ONET"))
)

merged["Target_Occupation"] = (
    merged["Target_SOC"].map(soc_title_dict)
    .fillna(merged.get("Target_Occupation_Lightcast"))
    .fillna(merged.get("Target_Occupation_ONET"))
)

merged = merged.merge(
    ai_lookup.rename(columns={
        "SOC": "Source_SOC",
        "AI_Exposure_Group": "Source_AI_Exposure_Group"
    }),
    on="Source_SOC",
    how="left"
)

merged = merged.merge(
    ai_lookup.rename(columns={
        "SOC": "Target_SOC",
        "AI_Exposure_Group": "Target_AI_Exposure_Group"
    }),
    on="Target_SOC",
    how="left"
)

merged = merged[
    merged["Source_AI_Exposure_Group"].isin(SOURCE_EXPOSURE_GROUPS)
].copy()

merged = merged.merge(
    ai_economic.rename(columns={
        "SOC": "Target_SOC",
        "AI_Occupation_Title": "AI_Target_Occupation_Title",
        "AI_Exposure_Group": "AI_Target_Exposure_Group"
    }),
    on="Target_SOC",
    how="left"
)

merged["Preferred_Annual_Openings"] = merged["AI_Annual_Total_Openings"].combine_first(
    merged["LC_Target_2024_2025_Estimated_Annual_Openings"]
)

merged["Preferred_Growth_Rate"] = merged["AI_Growth_Rate"]
merged["Preferred_Annual_Percent_Change"] = merged["AI_Annual_Percent_Change"]

merged["Preferred_Median_Annual_Wage"] = merged["AI_50th_Percentile_Wage"].combine_first(
    merged["LC_Target_Median_Annual_Earnings"]
)

merged["Preferred_Education"] = merged["AI_Education_Value"]
merged["Preferred_Work_Experience"] = merged["AI_Work_Experience_Value"]

# ============================================================
# NORMALIZED SCORING AND RANKS
# UPDATED HYBRID LIGHTCAST + O*NET METHODOLOGY
# ============================================================

merged["Lightcast_Compat_Score"] = merged["Final_Compat_Index"] / 100

already_decimal = merged["Final_Compat_Index"].between(0, 1, inclusive="both")

merged.loc[
    already_decimal,
    "Lightcast_Compat_Score"
] = merged.loc[
    already_decimal,
    "Final_Compat_Index"
]

def rank_score_by_source(df, rank_col):
    max_rank = df.groupby("Source_SOC")[rank_col].transform("max")

    out = 1 - ((df[rank_col] - 1) / (max_rank - 1))

    out = out.where(max_rank > 1, 1)

    return out

merged["Lightcast_Rank_Score"] = rank_score_by_source(
    merged,
    "Lightcast_Rank"
)

merged["ONET_Rank_Score"] = rank_score_by_source(
    merged,
    "ONET_Rank"
)

merged["ONET_Tier_Score"] = merged["ONET_Tier"].map({
    "Primary-Short": 1.00,
    "Primary-Long": 0.70,
    "Supplemental": 0.35
})

merged["Lightcast_Combined_Score"] = (
    0.60 * merged["Lightcast_Compat_Score"] +
    0.40 * merged["Lightcast_Rank_Score"]
)

merged["ONET_Combined_Score"] = (
    0.70 * merged["ONET_Tier_Score"] +
    0.30 * merged["ONET_Rank_Score"]
)

def final_score(row):
    lc = row["Lightcast_Combined_Score"]
    onet_score = row["ONET_Combined_Score"]
    source_type = str(row["Source_Type"]).strip()

    if source_type == "Both" and pd.notna(lc) and pd.notna(onet_score):
        score = (0.60 * lc) + (0.40 * onet_score)
        score += 0.10
        return min(score, 1.00)

    if source_type == "Lightcast" and pd.notna(lc):
        return lc

    if source_type == "ONET" and pd.notna(onet_score):
        return onet_score

    if pd.notna(lc) and pd.notna(onet_score):
        return (0.60 * lc) + (0.40 * onet_score)

    if pd.notna(lc):
        return lc

    if pd.notna(onet_score):
        return onet_score

    return np.nan

merged["Both_Combined_Score"] = merged.apply(final_score, axis=1)

merged["Both_Combined_Score_100"] = (
    merged["Both_Combined_Score"] * 100
).round(1)

merged["Both_Combined_Rank"] = (
    merged.groupby("Source_SOC")["Both_Combined_Score"]
    .rank(method="first", ascending=False)
)

merged["Both_Average_Source_Rank"] = (
    merged[["Lightcast_Rank", "ONET_Rank"]]
    .mean(axis=1)
)

merged = merged.sort_values(
    [
        "Source_SOC",
        "Both_Combined_Rank",
        "Both_Combined_Score",
        "Target_SOC"
    ],
    ascending=[True, True, False, True],
    na_position="last"
).copy()

# ============================================================
# FINAL COLUMN ORDER
# ============================================================

first_cols = [
    "Source_SOC",
    "Source_Occupation",
    "Source_AI_Exposure_Group",
    "Target_SOC",
    "Target_Occupation",
    "Target_AI_Exposure_Group",
    "Source_Type",
    "Lightcast_Rank",
    "ONET_Rank",
    "Both_Average_Source_Rank",
    "Both_Combined_Rank",
    "Final_Compat_Index",
    "Lightcast_Compat_Score",
    "Lightcast_Rank_Score",
    "Lightcast_Combined_Score",
    "ONET_Tier",
    "ONET_Index",
    "ONET_Tier_Score",
    "ONET_Rank_Score",
    "ONET_Combined_Score",
    "Both_Combined_Score",
    "Both_Combined_Score_100",
    "Preferred_Median_Annual_Wage",
    "Preferred_Annual_Openings",
    "Preferred_Annual_Percent_Change",
    "Preferred_Growth_Rate",
    "Preferred_Education",
    "Preferred_Work_Experience",
    "AI_Annual_Total_Openings",
    "AI_Annual_Percent_Change",
    "AI_Education_Value",
    "AI_Work_Experience_Value",
    "AI_Growth_Rate",
    "AI_50th_Percentile_Wage",
    "LC_Target_Median_Hourly_Earnings",
    "LC_Target_Median_Annual_Earnings",
    "LC_Target_2024_Jobs",
    "LC_Target_2025_Jobs",
    "LC_Target_2024_2025_Change",
    "LC_Target_2024_2025_Estimated_Annual_Openings",
    "Source_ONET_SOC_Lightcast",
    "Target_ONET_SOC_Lightcast",
    "Source_ONET_SOC_ONET",
    "Target_ONET_SOC_ONET",
    "Lightcast_Source_File"
]

existing_first = [c for c in first_cols if c in merged.columns]
remaining = [c for c in merged.columns if c not in existing_first]

final = merged[existing_first + remaining].sort_values(
    ["Source_SOC", "Both_Combined_Rank", "Target_SOC"],
    na_position="last"
).copy()

score_summary = (
    final.groupby("Source_Type", dropna=False)
    .agg(
        Rows=("Target_SOC", "size"),
        Avg_Lightcast_Compat_Score=("Lightcast_Compat_Score", "mean"),
        Avg_ONET_Combined_Score=("ONET_Combined_Score", "mean"),
        Avg_Both_Combined_Score=("Both_Combined_Score", "mean"),
        Min_Both_Combined_Score=("Both_Combined_Score", "min"),
        Max_Both_Combined_Score=("Both_Combined_Score", "max")
    )
    .reset_index()
)

source_type_by_source = (
    final.groupby(["Source_SOC", "Source_Occupation", "Source_Type"])
    .size()
    .reset_index(name="Rows")
)

target_not_in_ai = (
    final[final["Target_AI_Exposure_Group"].isna()]
    .drop_duplicates(subset=["Target_SOC", "Target_Occupation"])
    .sort_values("Target_SOC")
)
# ============================================================
# CLEAN DIRECT LOWER-EXPOSURE SHEET
# High/Very High sources can lead to Medium, Low, or Very Low
# Medium sources can only lead to Low or Very Low
# ============================================================

def allowed_direct_transition(row):
    source_exp = row["Source_AI_Exposure_Group"]
    target_exp = row["Target_AI_Exposure_Group"]

    if source_exp in ["High", "Very High"]:
        return target_exp in HIGH_VERY_HIGH_TARGET_GROUPS

    if source_exp == "Medium":
        return target_exp in MEDIUM_TARGET_GROUPS

    return False

clean_direct_lower = final[
    final.apply(allowed_direct_transition, axis=1)
].copy()

# ============================================================
# GATEWAY PATHWAYS
# Source occupation -> top 3 lower-exposure gateway occupations
# Gateway occupation -> additional lower-exposure destination occupations
# ============================================================

top_gateways = (
    clean_direct_lower
    .sort_values(
        ["Source_SOC", "Both_Combined_Rank", "Both_Combined_Score"],
        ascending=[True, True, False],
        na_position="last"
    )
    .groupby("Source_SOC")
    .head(3)
    .copy()
)

top_gateways["Gateway_Rank"] = top_gateways.groupby("Source_SOC").cumcount() + 1

# Load the full O*NET related occupations file again so gateways can be lower-exposure sources too
onet_gateway = pd.read_excel(ONET_FILE)
onet_gateway.columns = onet_gateway.columns.astype(str).str.strip()

onet_gateway = onet_gateway.rename(columns={
    "O*NET-SOC Code": "Gateway_ONET_SOC",
    "Title": "Gateway_Occupation_ONET",
    "Related O*NET-SOC Code": "Destination_ONET_SOC",
    "Related Title": "Destination_Occupation_ONET",
    "Relatedness Tier": "Destination_ONET_Tier",
    "Index": "Destination_ONET_Index"
})

onet_gateway["Gateway_ONET_SOC"] = onet_gateway["Gateway_ONET_SOC"].apply(fix_onet_detailed)
onet_gateway["Destination_ONET_SOC"] = onet_gateway["Destination_ONET_SOC"].apply(fix_onet_detailed)

onet_gateway["Gateway_SOC"] = onet_gateway.apply(
    lambda r: onet_to_soc(r["Gateway_ONET_SOC"], r["Gateway_Occupation_ONET"]),
    axis=1
)

onet_gateway["Destination_SOC"] = onet_gateway.apply(
    lambda r: onet_to_soc(r["Destination_ONET_SOC"], r["Destination_Occupation_ONET"]),
    axis=1
)

onet_gateway = onet_gateway.dropna(subset=["Gateway_SOC", "Destination_SOC"]).copy()

# Remove self-loops
onet_gateway = onet_gateway[
    onet_gateway["Gateway_SOC"] != onet_gateway["Destination_SOC"]
].copy()

onet_gateway["Destination_ONET_Index"] = onet_gateway["Destination_ONET_Index"].apply(clean_num)

onet_gateway["Destination_ONET_Tier_Score"] = onet_gateway["Destination_ONET_Tier"].map({
    "Primary-Short": 1.00,
    "Primary-Long": 0.75,
    "Supplemental": 0.50
})

tier_priority = {
    "Primary-Short": 1,
    "Primary-Long": 2,
    "Supplemental": 3
}

onet_gateway["Destination_ONET_Tier_Priority"] = (
    onet_gateway["Destination_ONET_Tier"].map(tier_priority)
)

onet_gateway = (
    onet_gateway
    .sort_values(
        [
            "Gateway_SOC",
            "Destination_SOC",
            "Destination_ONET_Tier_Priority",
            "Destination_ONET_Index"
        ],
        ascending=[True, True, True, True],
        na_position="last"
    )
    .drop_duplicates(subset=["Gateway_SOC", "Destination_SOC"], keep="first")
    .copy()
)

# Add destination exposure
onet_gateway = onet_gateway.merge(
    ai_lookup.rename(columns={
        "SOC": "Destination_SOC",
        "AI_Exposure_Group": "Destination_AI_Exposure_Group"
    }),
    on="Destination_SOC",
    how="left"
)

# Keep only possible lower-exposure gateway destinations.
# Final source-specific filtering happens after gateway paths are created.
onet_gateway = onet_gateway[
    onet_gateway["Destination_AI_Exposure_Group"].isin(["Very Low", "Low", "Medium"])
].copy()

# Add destination title and AI economic data
onet_gateway["Destination_Occupation"] = (
    onet_gateway["Destination_SOC"]
    .map(soc_title_dict)
    .fillna(onet_gateway["Destination_Occupation_ONET"])
)

onet_gateway = onet_gateway.merge(
    ai_economic.rename(columns={
        "SOC": "Destination_SOC",
        "AI_Occupation_Title": "AI_Destination_Occupation_Title",
        "AI_Exposure_Group": "AI_Destination_Exposure_Group"
    }),
    on="Destination_SOC",
    how="left"
)

# Rank gateway destinations within each gateway occupation
onet_gateway["Gateway_Destination_Rank"] = (
    onet_gateway
    .groupby("Gateway_SOC")["Destination_ONET_Index"]
    .rank(method="first", ascending=True)
)

# Join top gateways to their lower-exposure destinations
gateway_paths = top_gateways.merge(
    onet_gateway,
    left_on="Target_SOC",
    right_on="Gateway_SOC",
    how="left",
    suffixes=("", "_Gateway")
)

gateway_paths = gateway_paths[
    gateway_paths["Destination_SOC"].notna()
].copy()

# Final source-specific gateway filter:
# High/Very High sources can lead to Medium, Low, or Very Low.
# Medium sources can only lead to Low or Very Low.
def allowed_gateway_transition(row):
    source_exp = row["Source_AI_Exposure_Group"]
    dest_exp = row["Destination_AI_Exposure_Group"]

    if source_exp in ["High", "Very High"]:
        return dest_exp in HIGH_VERY_HIGH_TARGET_GROUPS

    if source_exp == "Medium":
        return dest_exp in MEDIUM_TARGET_GROUPS

    return False

gateway_paths = gateway_paths[
    gateway_paths.apply(allowed_gateway_transition, axis=1)
].copy()

# Remove cases where destination loops back to original source
gateway_paths = gateway_paths[
    gateway_paths["Destination_SOC"] != gateway_paths["Source_SOC"]
].copy()

# Remove cases where destination is same as gateway
gateway_paths = gateway_paths[
    gateway_paths["Destination_SOC"] != gateway_paths["Gateway_SOC"]
].copy()

gateway_paths["Pathway_Type"] = "Indirect / Gateway"

gateway_paths["Gateway_Occupation"] = gateway_paths["Target_Occupation"]
gateway_paths["Gateway_AI_Exposure_Group"] = gateway_paths["Target_AI_Exposure_Group"]

gateway_paths["Gateway_Pathway_Label"] = (
    gateway_paths["Source_Occupation"].astype(str)
    + " -> "
    + gateway_paths["Gateway_Occupation"].astype(str)
    + " -> "
    + gateway_paths["Destination_Occupation"].astype(str)
)

gateway_paths = gateway_paths.sort_values(
    [
        "Source_SOC",
        "Gateway_Rank",
        "Gateway_Destination_Rank",
        "Destination_ONET_Tier_Priority",
        "Destination_ONET_Index"
    ],
    ascending=[True, True, True, True, True],
    na_position="last"
).copy()

clean_gateway_paths = gateway_paths[[
    "Pathway_Type",
    "Gateway_Pathway_Label",

    "Source_SOC",
    "Source_Occupation",
    "Source_AI_Exposure_Group",

    "Gateway_Rank",
    "Gateway_SOC",
    "Gateway_Occupation",
    "Gateway_AI_Exposure_Group",
    "Source_Type",
    "Lightcast_Rank",
    "ONET_Rank",
    "Both_Combined_Rank",
    "Both_Combined_Score_100",

    "Destination_SOC",
    "Destination_Occupation",
    "Destination_AI_Exposure_Group",
    "Destination_ONET_Tier",
    "Destination_ONET_Index",
    "Destination_ONET_Tier_Score",
    "Gateway_Destination_Rank",

    "AI_Annual_Total_Openings",
    "AI_Annual_Percent_Change",
    "AI_Education_Value",
    "AI_Work_Experience_Value",
    "AI_Growth_Rate",
    "AI_50th_Percentile_Wage"
]].copy()

# ============================================================
# RELATED OCCUPATIONS FINAL SHEET
# Direct + Indirect pathways to Very Low, Low, and Medium only
# ============================================================

# ----------------------------
# DIRECT PATHWAYS
# Source occupation -> Related occupation
# ----------------------------

direct_related = clean_direct_lower.copy()

direct_related["Pathway_Type"] = "Direct"
direct_related["Gateway_SOC"] = ""
direct_related["Gateway_Occupation"] = ""
direct_related["Gateway_AI_Exposure_Group"] = ""
direct_related["Gateway_Rank"] = np.nan

# For direct rows, O*NET rank/index/tier describe Source -> Related Occupation
direct_related["Related_ONET_Rank"] = direct_related["ONET_Rank"]
direct_related["Related_ONET_Index"] = direct_related["ONET_Index"]
direct_related["Related_ONET_Tier"] = direct_related["ONET_Tier"]
direct_related["Related_ONET_Score"] = direct_related["ONET_Combined_Score"]

direct_related = direct_related.rename(columns={
    "Pathway_Type": "Pathway Type",
    "Source_SOC": "Source SOC",
    "Source_Occupation": "Source Occupation",
    "Source_AI_Exposure_Group": "Source AI Exposure",
    "Gateway_SOC": "Gateway SOC",
    "Gateway_Occupation": "Gateway Occupation",
    "Gateway_AI_Exposure_Group": "Gateway AI Exposure",
    "Gateway_Rank": "Gateway Rank",
    "Target_SOC": "Related Occupation SOC",
    "Target_Occupation": "Related Occupation",
    "Target_AI_Exposure_Group": "Related Occupation AI Exposure",
    "Source_Type": "Relationship Source",
    "Lightcast_Rank": "Lightcast Rank",
    "Both_Average_Source_Rank": "Average Source Rank",
    "Both_Combined_Rank": "Combined Rank",
    "Final_Compat_Index": "Lightcast Compatibility Index",
    "Lightcast_Combined_Score": "Lightcast Score",
    "Both_Combined_Score": "Combined Lightcast + O*NET Score",
    "Both_Combined_Score_100": "Combined Score 0-100",
    "Related_ONET_Rank": "Related Occupation O*NET Rank",
    "Related_ONET_Index": "Related Occupation O*NET Index",
    "Related_ONET_Tier": "Related Occupation O*NET Tier",
    "Related_ONET_Score": "Related Occupation O*NET Score"
})

direct_related = direct_related[[
    "Pathway Type",
    "Source SOC",
    "Source Occupation",
    "Source AI Exposure",
    "Gateway Rank",
    "Gateway SOC",
    "Gateway Occupation",
    "Gateway AI Exposure",
    "Related Occupation SOC",
    "Related Occupation",
    "Related Occupation AI Exposure",
    "Relationship Source",
    "Lightcast Rank",
    "Related Occupation O*NET Rank",
    "Average Source Rank",
    "Combined Rank",
    "Lightcast Compatibility Index",
    "Related Occupation O*NET Tier",
    "Related Occupation O*NET Index",
    "Lightcast Score",
    "Related Occupation O*NET Score",
    "Combined Lightcast + O*NET Score",
    "Combined Score 0-100"
]].copy()

# ----------------------------
# INDIRECT / GATEWAY PATHWAYS
# Source occupation -> Gateway occupation -> Related occupation
# ----------------------------

indirect_related = clean_gateway_paths.copy()

# For indirect rows, O*NET rank/index/tier describe Gateway -> Related Occupation
indirect_related["Related_ONET_Rank"] = indirect_related["Gateway_Destination_Rank"]
indirect_related["Related_ONET_Index"] = indirect_related["Destination_ONET_Index"]
indirect_related["Related_ONET_Tier"] = indirect_related["Destination_ONET_Tier"]
indirect_related["Related_ONET_Score"] = indirect_related["Destination_ONET_Tier_Score"]

# For indirect rows, the original source-to-gateway score is shown only as the gateway score
indirect_related["Source_to_Gateway_Combined_Rank"] = indirect_related["Both_Combined_Rank"]
indirect_related["Source_to_Gateway_Combined_Score_100"] = indirect_related["Both_Combined_Score_100"]

indirect_related = indirect_related.rename(columns={
    "Pathway_Type": "Pathway Type",
    "Source_SOC": "Source SOC",
    "Source_Occupation": "Source Occupation",
    "Source_AI_Exposure_Group": "Source AI Exposure",
    "Gateway_Rank": "Gateway Rank",
    "Gateway_SOC": "Gateway SOC",
    "Gateway_Occupation": "Gateway Occupation",
    "Gateway_AI_Exposure_Group": "Gateway AI Exposure",
    "Destination_SOC": "Related Occupation SOC",
    "Destination_Occupation": "Related Occupation",
    "Destination_AI_Exposure_Group": "Related Occupation AI Exposure",
    "Source_Type": "Relationship Source",
    "Related_ONET_Rank": "Related Occupation O*NET Rank",
    "Related_ONET_Index": "Related Occupation O*NET Index",
    "Related_ONET_Tier": "Related Occupation O*NET Tier",
    "Related_ONET_Score": "Related Occupation O*NET Score"
})

# For indirect rows, Lightcast Rank is intentionally blank because we are not ranking
# Gateway -> Related Occupation using Lightcast in this script.
indirect_related["Lightcast Rank"] = np.nan
indirect_related["Average Source Rank"] = np.nan
indirect_related["Combined Rank"] = indirect_related["Source_to_Gateway_Combined_Rank"]
indirect_related["Lightcast Compatibility Index"] = np.nan
indirect_related["Lightcast Score"] = np.nan
indirect_related["Combined Lightcast + O*NET Score"] = np.nan
indirect_related["Combined Score 0-100"] = indirect_related["Source_to_Gateway_Combined_Score_100"]

indirect_related = indirect_related[[
    "Pathway Type",
    "Source SOC",
    "Source Occupation",
    "Source AI Exposure",
    "Gateway Rank",
    "Gateway SOC",
    "Gateway Occupation",
    "Gateway AI Exposure",
    "Related Occupation SOC",
    "Related Occupation",
    "Related Occupation AI Exposure",
    "Relationship Source",
    "Lightcast Rank",
    "Related Occupation O*NET Rank",
    "Average Source Rank",
    "Combined Rank",
    "Lightcast Compatibility Index",
    "Related Occupation O*NET Tier",
    "Related Occupation O*NET Index",
    "Lightcast Score",
    "Related Occupation O*NET Score",
    "Combined Lightcast + O*NET Score",
    "Combined Score 0-100"
]].copy()

# ----------------------------
# COMBINE DIRECT + INDIRECT
# ----------------------------

related_occupations_final = pd.concat(
    [direct_related, indirect_related],
    ignore_index=True
)

# Direct pathways are always listed before gateway pathways.
related_occupations_final["Pathway Sort"] = related_occupations_final["Pathway Type"].map({
    "Direct": 1,
    "Indirect / Gateway": 2
})

related_occupations_final = related_occupations_final.sort_values(
    [
        "Source SOC",
        "Pathway Sort",
        "Combined Rank",
        "Gateway Rank",
        "Related Occupation O*NET Rank",
        "Combined Score 0-100",
        "Related Occupation SOC"
    ],
    ascending=[True, True, True, True, True, False, True],
    na_position="last"
).copy()

related_occupations_final = related_occupations_final.drop(
    columns=["Pathway Sort"],
    errors="ignore"
)

# ============================================================
# EXPORT
# ============================================================

with pd.ExcelWriter(OUT_FILE, engine="openpyxl") as writer:
    related_occupations_final.to_excel(writer, sheet_name="Related_Occs_Final", index=False)
    final.to_excel(writer, sheet_name="Final_Transitions", index=False)
    clean_direct_lower.to_excel(writer, sheet_name="Clean_Direct_Lower_Only", index=False)
    clean_gateway_paths.to_excel(writer, sheet_name="Clean_Gateway_Pathways", index=False)
    lc.to_excel(writer, sheet_name="Lightcast_Parsed", index=False)
    onet.to_excel(writer, sheet_name="ONET_Parsed", index=False)
    lc_raw.to_excel(writer, sheet_name="Lightcast_Raw_All", index=False)
    lc_source_econ.to_excel(writer, sheet_name="Lightcast_Source_Econ", index=False)
    lc_source_edu.to_excel(writer, sheet_name="Lightcast_Source_Edu", index=False)
    ai_economic.to_excel(writer, sheet_name="AI_Economic", index=False)
    score_summary.to_excel(writer, sheet_name="Score_Summary", index=False)
    source_type_by_source.to_excel(writer, sheet_name="Source_Type_By_Source", index=False)
    target_not_in_ai.to_excel(writer, sheet_name="Target_Not_In_AI", index=False)
    lc_errors.to_excel(writer, sheet_name="Lightcast_Parse_Errors", index=False)
    soc_title_lookup.to_excel(writer, sheet_name="SOC_Title_Lookup", index=False)
    crosswalk.to_excel(writer, sheet_name="Crosswalk_Used", index=False)

# ============================================================
# FORMAT WORKBOOK
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = load_workbook(OUT_FILE)

teal_fill = PatternFill("solid", fgColor="17C3B2")
light_fill = PatternFill("solid", fgColor="F2F2F2")
white_fill = PatternFill("solid", fgColor="FFFFFF")
white_font = Font(color="FFFFFF", bold=True)
body_font = Font(color="111111", size=10)
thin_border = Border(
    left=Side(style="thin", color="D9D9D9"),
    right=Side(style="thin", color="D9D9D9"),
    top=Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9")
)

for ws in wb.worksheets:
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    for cell in ws[1]:
        cell.fill = teal_fill
        cell.font = white_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = thin_border

    for row in ws.iter_rows(min_row=2):
        fill = light_fill if row[0].row % 2 == 0 else white_fill

        for cell in row:
            cell.fill = fill
            cell.font = body_font
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            cell.border = thin_border

    for col_idx, col_cells in enumerate(ws.columns, start=1):
        max_len = 0

        for cell in col_cells:
            if cell.value is not None:
                max_len = max(max_len, len(str(cell.value)))

        width = min(max(max_len + 2, 10), 42)
        ws.column_dimensions[get_column_letter(col_idx)].width = width

    ws.row_dimensions[1].height = 42

    for row_idx in range(2, ws.max_row + 1):
        ws.row_dimensions[row_idx].height = 24

# Specific formatting for final related occupations sheet
ws = wb["Related_Occs_Final"]

percent_cols = [
    "Lightcast Score",
    "Related Occupation O*NET Score",
    "Combined Lightcast + O*NET Score"
]

score_cols = [
    "Combined Score 0-100",
    "Lightcast Compatibility Index"
]

rank_cols = [
    "Gateway Rank",
    "Lightcast Rank",
    "Related Occupation O*NET Rank",
    "Average Source Rank",
    "Combined Rank",
    "Related Occupation O*NET Index"
]

for col_idx, cell in enumerate(ws[1], start=1):
    if cell.value in percent_cols:
        for row in range(2, ws.max_row + 1):
            ws.cell(row=row, column=col_idx).number_format = "0.0%"

    if cell.value in score_cols:
        for row in range(2, ws.max_row + 1):
            ws.cell(row=row, column=col_idx).number_format = "0.0"

    if cell.value in rank_cols:
        for row in range(2, ws.max_row + 1):
            ws.cell(row=row, column=col_idx).number_format = "0"

wb.save(OUT_FILE)

# ============================================================
# SAVE MAIN RELATED OCCUPATIONS SHEET AS CSV
# ============================================================

CSV_OUT_FILE = OUT_FILE.with_name(
    OUT_FILE.stem + "_Related_Occs_Final.csv"
)

related_occupations_final.to_csv(
    CSV_OUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("Saved CSV:")
print(CSV_OUT_FILE)

print("Saved rebuilt workbook:")
print(OUT_FILE)

print("\nRelated occupations final rows:", related_occupations_final.shape)
print("\nFinal rows:", final.shape)
print("\nSource type summary:")
print(final["Source_Type"].value_counts(dropna=False))

Crosswalk rows: 1016
Medium + High + Very High source SOCs: 519
Parsed Lightcast rows: 2665
Lightcast source SOCs: 270
O*NET rows: 9254
O*NET source SOCs: 460
Saved CSV:
C:\Users\301533\Documents\AI Analysis\Ligthcast Skills & RO\FINAL_REBUILT_High_VeryHigh_AND_Medium_Related_Occs_20260513_131923_Related_Occs_Final.csv
Saved rebuilt workbook:
C:\Users\301533\Documents\AI Analysis\Ligthcast Skills & RO\FINAL_REBUILT_High_VeryHigh_AND_Medium_Related_Occs_20260513_131923.xlsx

Related occupations final rows: (12931, 23)

Final rows: (10402, 50)

Source type summary:
Source_Type
ONET         7737
Both         1517
Lightcast    1148
Name: count, dtype: int64


In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from datetime import datetime

# ============================================================
# PATHS
# ============================================================

LIGHTCAST_HIGH_FOLDER = Path(
    r"C:\Users\301533\Documents\AI Analysis\Ligthcast Skills & RO\High AI"
)

LIGHTCAST_VERY_HIGH_FOLDER = Path(
    r"C:\Users\301533\Documents\AI Analysis\Ligthcast Skills & RO"
)

ONET_FILE = Path(
    r"C:\Users\301533\Downloads\Related Occupations (2).xlsx"
)

CROSSWALK_FILE = Path(
    r"C:\Users\301533\Documents\AI Analysis\2019_to_SOC_Crosswalk.xlsx"
)

AI_FILE = Path(
    r"G:\Shared drives\EO_ECONOMIC ANALYSIS\ea_common\Projects\AI\risk_index\ai_exposure_composite_clean.csv"
)

OUT_FILE = Path(
    rf"C:\Users\301533\Documents\AI Analysis\Ligthcast Skills & RO\FINAL_REBUILT_High_VeryHigh_AND_Medium_Related_Occs_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
)

SOURCE_EXPOSURE_GROUPS = ["Medium", "High", "Very High"]

HIGH_VERY_HIGH_TARGET_GROUPS = ["Very Low", "Low", "Medium"]
MEDIUM_TARGET_GROUPS = ["Very Low", "Low"]

# ============================================================
# HELPERS
# ============================================================

month_map = {
    "Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04",
    "May": "05", "Jun": "06", "Jul": "07", "Aug": "08",
    "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"
}

def clean_num(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    if s in ["", "-", "NA", "N/A", "nan", "None"]:
        return np.nan

    neg = False
    if s.startswith("(") and s.endswith(")"):
        neg = True
        s = s.replace("(", "").replace(")", "")

    s = (
        s.replace("$", "")
         .replace("/hr", "")
         .replace(",", "")
         .replace("%", "")
         .replace("−", "-")
         .strip()
    )

    out = pd.to_numeric(s, errors="coerce")

    if pd.notna(out) and neg:
        out = -out

    return out

def clean_title(x):
    if pd.isna(x):
        return np.nan

    return (
        str(x)
        .strip()
        .lower()
        .replace("*", "")
        .replace(",", "")
        .replace("  ", " ")
    )

def fix_soc_basic(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip().replace(".0", "")

    if s.lower() in ["nan", "none", ""]:
        return np.nan

    match = re.search(r"(\d{2})-(\d{4})", s)
    if match:
        return f"{match.group(1)}-{match.group(2)}"

    if "-" in s:
        left, right = s.split("-", 1)

        if left in month_map and right.isdigit():
            if len(right) <= 2:
                return f"{month_map[left]}-20{right.zfill(2)}"
            return f"{month_map[left]}-{right.zfill(4)}"

        if right in month_map and left.isdigit():
            return f"{month_map[right]}-20{left.zfill(2)}"

    return s

def fix_onet_detailed(x):
    if pd.isna(x):
        return np.nan

    s = str(x).strip()

    match = re.search(r"(\d{2}-\d{4}(?:\.\d{2})?)", s)
    if match:
        return match.group(1)

    return fix_soc_basic(s)

def extract_occ_and_onet(value):
    if pd.isna(value):
        return np.nan, np.nan

    s = str(value).strip()

    match = re.search(r"^(.*?)\s*\((\d{2}-\d{4}(?:\.\d{2})?)\)", s)

    if match:
        return match.group(1).strip(), match.group(2).strip()

    return s, np.nan

def get_parameters_source(file):
    params = pd.read_excel(file, sheet_name="Parameters", header=None)

    source_onet = np.nan
    source_title = np.nan

    for i in range(len(params)):
        vals = params.iloc[i].astype(str).str.strip().tolist()

        if len(vals) >= 2 and vals[0] == "Code" and vals[1] == "Description":
            if i + 1 < len(params):
                source_onet = params.iloc[i + 1, 0]
                source_title = params.iloc[i + 1, 1]
                break

    return fix_onet_detailed(source_onet), source_title

# ============================================================
# LOAD CROSSWALK
# ============================================================

crosswalk = pd.read_excel(CROSSWALK_FILE, header=3)
crosswalk.columns = crosswalk.columns.astype(str).str.strip()

crosswalk = crosswalk.rename(columns={
    "O*NET-SOC 2019 Code": "ONET_2019_Code",
    "O*NET-SOC 2019 Title": "ONET_2019_Title",
    "2018 SOC Code": "SOC_2018",
    "2018 SOC Title": "SOC_2018_Title"
})

crosswalk["ONET_2019_Code_Clean"] = crosswalk["ONET_2019_Code"].apply(fix_onet_detailed)
crosswalk["SOC_2018_Clean"] = crosswalk["SOC_2018"].apply(fix_soc_basic)
crosswalk["ONET_Title_Clean"] = crosswalk["ONET_2019_Title"].apply(clean_title)

onet_to_soc2018 = (
    crosswalk
    .dropna(subset=["ONET_2019_Code_Clean", "SOC_2018_Clean"])
    .drop_duplicates(subset=["ONET_2019_Code_Clean"])
    .set_index("ONET_2019_Code_Clean")["SOC_2018_Clean"]
    .to_dict()
)

title_to_soc2018 = (
    crosswalk
    .dropna(subset=["ONET_Title_Clean", "SOC_2018_Clean"])
    .drop_duplicates(subset=["ONET_Title_Clean"])
    .set_index("ONET_Title_Clean")["SOC_2018_Clean"]
    .to_dict()
)

soc_title_lookup = (
    crosswalk[["SOC_2018_Clean", "SOC_2018_Title"]]
    .dropna()
    .drop_duplicates(subset=["SOC_2018_Clean"])
    .copy()
)

soc_title_dict = dict(
    zip(soc_title_lookup["SOC_2018_Clean"], soc_title_lookup["SOC_2018_Title"])
)

def onet_to_soc(onet_code, title=None):
    onet_clean = fix_onet_detailed(onet_code)

    if pd.notna(onet_clean) and onet_clean in onet_to_soc2018:
        return onet_to_soc2018[onet_clean]

    title_clean = clean_title(title)
    if pd.notna(title_clean) and title_clean in title_to_soc2018:
        return title_to_soc2018[title_clean]

    return fix_soc_basic(onet_clean)

print("Crosswalk rows:", len(crosswalk))

# ============================================================
# LOAD AI COMPOSITE
# ============================================================

ai = pd.read_csv(AI_FILE)
ai.columns = ai.columns.astype(str).str.strip()

possible_soc_cols = [
    "SOC", "SOC Code", "SOC_Code", "SOC2018", "SOC2018.1",
    "O*NET-SOC Code", "O*NET_SOC_Code", "O*NET-SOC Code.1"
]

possible_title_cols = [
    "Occupation Title", "Title", "Occupation", "2018 SOC Title", "O*NET-SOC 2019 Title"
]

possible_exposure_cols = [
    "AI Exposure Category", "AI_Exposure_Group", "exposure", "Exposure",
    "AI Exposure Group", "AI_Exposure_Category", "Exposure_Group"
]

soc_col = next((c for c in possible_soc_cols if c in ai.columns), None)
title_col = next((c for c in possible_title_cols if c in ai.columns), None)
exposure_col = next((c for c in possible_exposure_cols if c in ai.columns), None)

if soc_col is None:
    raise ValueError("Could not find SOC column in AI composite.")

if title_col is None:
    raise ValueError("Could not find title column in AI composite.")

if exposure_col is None:
    raise ValueError("Could not find exposure column in AI composite.")

ai["Title_Clean"] = ai[title_col].apply(clean_title)
ai["SOC_From_Title"] = ai["Title_Clean"].map(title_to_soc2018)
ai["SOC_Fallback"] = ai[soc_col].apply(fix_soc_basic)
ai["SOC"] = ai["SOC_From_Title"].fillna(ai["SOC_Fallback"])
ai["AI_Exposure_Group"] = ai[exposure_col].astype(str).str.strip()

ai_lookup = (
    ai[["SOC", "AI_Exposure_Group"]]
    .dropna()
    .drop_duplicates(subset=["SOC"])
    .copy()
)

source_socs = set(
    ai_lookup.loc[
        ai_lookup["AI_Exposure_Group"].isin(SOURCE_EXPOSURE_GROUPS),
        "SOC"
    ]
)

ai_econ_cols = [
    "SOC",
    title_col,
    "AI_Exposure_Group",
    "Annual Total Openings",
    "Annual Percent Change",
    "Education Value",
    "Work Experience Value",
    "Growth Rate",
    "50th Percentile Wage"
]

ai_econ_cols = [c for c in ai_econ_cols if c in ai.columns]

ai_economic = ai[ai_econ_cols].copy()

ai_economic = ai_economic.rename(columns={
    title_col: "AI_Occupation_Title",
    "Annual Total Openings": "AI_Annual_Total_Openings",
    "Annual Percent Change": "AI_Annual_Percent_Change",
    "Education Value": "AI_Education_Value",
    "Work Experience Value": "AI_Work_Experience_Value",
    "Growth Rate": "AI_Growth_Rate",
    "50th Percentile Wage": "AI_50th_Percentile_Wage"
})

ai_economic = ai_economic.drop_duplicates(subset=["SOC"])

print("Medium + High + Very High source SOCs:", len(source_socs))

# ============================================================
# PARSE LIGHTCAST INDIVIDUAL FILES
# ============================================================

lightcast_files = []
lightcast_files += list(LIGHTCAST_HIGH_FOLDER.glob("Skills_Transferability_*.xlsx"))
lightcast_files += list(LIGHTCAST_VERY_HIGH_FOLDER.glob("Skills_Transferability_*.xlsx"))
lightcast_files = sorted(set(lightcast_files))

lc_rows = []
lc_source_econ_rows = []
lc_source_edu_rows = []
lc_errors = []

for file in lightcast_files:
    try:
        xls = pd.ExcelFile(file)

        if "Parameters" not in xls.sheet_names:
            continue

        source_onet, source_title = get_parameters_source(file)
        source_soc = onet_to_soc(source_onet, source_title)

        if pd.isna(source_soc):
            lc_errors.append({
                "File": file.name,
                "Error": "Could not identify source SOC from Parameters"
            })
            continue

        if "Overview" in xls.sheet_names:
            overview = pd.read_excel(file, sheet_name="Overview", header=None)
            overview_metrics = {}

            for _, row in overview.iterrows():
                label = row.iloc[0] if len(row) > 0 else np.nan
                value = row.iloc[1] if len(row) > 1 else np.nan

                if isinstance(label, str):
                    label_clean = label.strip()
                    overview_metrics[label_clean] = value

            source_median_hourly = clean_num(overview_metrics.get("Median Hourly Earnings*"))

            lc_source_econ_rows.append({
                "Source_SOC": source_soc,
                "Source_ONET_SOC": source_onet,
                "Source_Occupation": source_title,
                "LC_Source_Median_Hourly_Earnings": source_median_hourly,
                "LC_Source_Median_Annual_Earnings": source_median_hourly * 2080 if pd.notna(source_median_hourly) else np.nan,
                "LC_Source_2025_Jobs": clean_num(overview_metrics.get("2025 Jobs*")),
                "LC_Source_2024_2025_Change": clean_num(overview_metrics.get("2024-2025 Change*")),
                "LC_Source_2024_2025_Estimated_Annual_Openings": clean_num(overview_metrics.get("2024-2025 Estimated Annual Openings*")),
                "Source_File": file.name
            })

        if "Edu Attainment" in xls.sheet_names:
            edu = pd.read_excel(file, sheet_name="Edu Attainment", header=None)
            edu = edu.dropna(how="all")

            for _, row in edu.iterrows():
                edu_level = row.iloc[0] if len(row) > 0 else np.nan
                edu_pct = row.iloc[1] if len(row) > 1 else np.nan

                if isinstance(edu_level, str):
                    edu_level = edu_level.strip()

                    if edu_level not in ["Educational Attainment", ""]:
                        pct = clean_num(edu_pct)

                        lc_source_edu_rows.append({
                            "Source_SOC": source_soc,
                            "Source_ONET_SOC": source_onet,
                            "Source_Occupation": source_title,
                            "LC_Source_Education_Level": edu_level,
                            "LC_Source_Education_Percent": pct / 100 if pd.notna(pct) and pct > 1 else pct,
                            "Source_File": file.name
                        })

        compatible_sheet = None

        for s in xls.sheet_names:
            if "top compatible" in str(s).lower().strip():
                compatible_sheet = s
                break

        if compatible_sheet is None:
            lc_errors.append({
                "File": file.name,
                "Source_SOC": source_soc,
                "Error": "Missing Top Compatible Occs sheet",
                "Sheets": " | ".join(xls.sheet_names)
            })
            continue

        comp = pd.read_excel(
            file,
            sheet_name=compatible_sheet,
            header=2
        )

        comp.columns = comp.columns.astype(str).str.strip()
        comp = comp.dropna(how="all")

        comp = comp.rename(columns={
            "O*NET Occupation": "Target_ONET_Occupation_Raw",
            "Median Hourly Earnings*": "LC_Target_Median_Hourly_Earnings",
            "2024 Jobs*": "LC_Target_2024_Jobs",
            "2025 Jobs*": "LC_Target_2025_Jobs",
            "2024-2025 Change*": "LC_Target_2024_2025_Change",
            "2024-2025 Estimated Annual Openings*": "LC_Target_2024_2025_Estimated_Annual_Openings",
            "Compat. Index": "Lightcast_Compat_Index"
        })

        if "Target_ONET_Occupation_Raw" not in comp.columns:
            lc_errors.append({
                "File": file.name,
                "Source_SOC": source_soc,
                "Sheet": compatible_sheet,
                "Error": "Missing target occupation column",
                "Columns": " | ".join(comp.columns)
            })
            continue

        for _, row in comp.iterrows():
            target_title, target_onet = extract_occ_and_onet(
                row.get("Target_ONET_Occupation_Raw")
            )

            target_soc = onet_to_soc(target_onet, target_title)

            if pd.isna(target_soc):
                continue

            if source_soc == target_soc:
                continue

            median_hourly = clean_num(row.get("LC_Target_Median_Hourly_Earnings"))

            lc_rows.append({
                "Source_SOC": source_soc,
                "Source_ONET_SOC": source_onet,
                "Source_Occupation": source_title,
                "Target_SOC": target_soc,
                "Target_ONET_SOC": target_onet,
                "Target_Occupation_Lightcast": target_title,
                "Final_Compat_Index": clean_num(row.get("Lightcast_Compat_Index")),
                "LC_Target_Median_Hourly_Earnings": median_hourly,
                "LC_Target_Median_Annual_Earnings": median_hourly * 2080 if pd.notna(median_hourly) else np.nan,
                "LC_Target_2024_Jobs": clean_num(row.get("LC_Target_2024_Jobs")),
                "LC_Target_2025_Jobs": clean_num(row.get("LC_Target_2025_Jobs")),
                "LC_Target_2024_2025_Change": clean_num(row.get("LC_Target_2024_2025_Change")),
                "LC_Target_2024_2025_Estimated_Annual_Openings": clean_num(row.get("LC_Target_2024_2025_Estimated_Annual_Openings")),
                "Lightcast_Source_File": file.name
            })

    except Exception as e:
        lc_errors.append({
            "File": file.name,
            "Error": str(e)
        })

lc_raw = pd.DataFrame(lc_rows)
lc_source_econ = pd.DataFrame(lc_source_econ_rows)
lc_source_edu = pd.DataFrame(lc_source_edu_rows)
lc_errors = pd.DataFrame(lc_errors)

if lc_raw.empty:
    raise ValueError("No Lightcast compatible occupation rows were parsed.")

lc_raw["Source_SOC"] = lc_raw["Source_SOC"].apply(fix_soc_basic)
lc_raw["Target_SOC"] = lc_raw["Target_SOC"].apply(fix_soc_basic)

lc_raw = lc_raw[lc_raw["Source_SOC"].isin(source_socs)].copy()

lc = (
    lc_raw
    .sort_values(["Source_SOC", "Target_SOC", "Final_Compat_Index"], ascending=[True, True, False])
    .drop_duplicates(subset=["Source_SOC", "Target_SOC"], keep="first")
    .copy()
)

lc["Lightcast_Rank"] = (
    lc.groupby("Source_SOC")["Final_Compat_Index"]
    .rank(method="first", ascending=False)
)

print("Parsed Lightcast rows:", len(lc))
print("Lightcast source SOCs:", lc["Source_SOC"].nunique())

# ============================================================
# LOAD ONET RELATED OCCUPATIONS
# ============================================================

onet = pd.read_excel(ONET_FILE)
onet.columns = onet.columns.astype(str).str.strip()

onet = onet.rename(columns={
    "O*NET-SOC Code": "Source_ONET_SOC",
    "Title": "Source_Occupation_ONET",
    "Related O*NET-SOC Code": "Target_ONET_SOC",
    "Related Title": "Target_Occupation_ONET",
    "Relatedness Tier": "ONET_Tier",
    "Index": "ONET_Index"
})

onet["Source_ONET_SOC"] = onet["Source_ONET_SOC"].apply(fix_onet_detailed)
onet["Target_ONET_SOC"] = onet["Target_ONET_SOC"].apply(fix_onet_detailed)

onet["Source_SOC"] = onet.apply(
    lambda r: onet_to_soc(r["Source_ONET_SOC"], r["Source_Occupation_ONET"]),
    axis=1
)

onet["Target_SOC"] = onet.apply(
    lambda r: onet_to_soc(r["Target_ONET_SOC"], r["Target_Occupation_ONET"]),
    axis=1
)

onet = onet.dropna(subset=["Source_SOC", "Target_SOC"]).copy()
onet = onet[onet["Source_SOC"].isin(source_socs)].copy()
onet = onet[onet["Source_SOC"] != onet["Target_SOC"]].copy()

tier_priority = {
    "Primary-Short": 1,
    "Primary-Long": 2,
    "Supplemental": 3
}

onet["ONET_Tier_Priority"] = onet["ONET_Tier"].map(tier_priority)
onet["ONET_Index"] = onet["ONET_Index"].apply(clean_num)

onet = (
    onet
    .sort_values(["Source_SOC", "Target_SOC", "ONET_Tier_Priority", "ONET_Index"])
    .drop_duplicates(subset=["Source_SOC", "Target_SOC"], keep="first")
    .copy()
)

onet["ONET_Rank"] = (
    onet.groupby("Source_SOC")["ONET_Index"]
    .rank(method="first", ascending=True)
)

print("O*NET rows:", len(onet))
print("O*NET source SOCs:", onet["Source_SOC"].nunique())

# ============================================================
# MERGE LIGHTCAST + ONET
# ============================================================

merged = lc.merge(
    onet[
        [
            "Source_SOC",
            "Target_SOC",
            "Source_ONET_SOC",
            "Target_ONET_SOC",
            "Source_Occupation_ONET",
            "Target_Occupation_ONET",
            "ONET_Tier",
            "ONET_Index",
            "ONET_Rank"
        ]
    ],
    on=["Source_SOC", "Target_SOC"],
    how="outer",
    suffixes=("_Lightcast", "_ONET")
)

merged["Source_Type"] = np.select(
    [
        merged["Final_Compat_Index"].notna() & merged["ONET_Tier"].notna(),
        merged["Final_Compat_Index"].notna(),
        merged["ONET_Tier"].notna()
    ],
    [
        "Both",
        "Lightcast",
        "ONET"
    ],
    default="Unknown"
)

# ============================================================
# TITLES, EXPOSURE, ECONOMIC DATA
# ============================================================

merged["Source_Occupation"] = (
    merged["Source_SOC"].map(soc_title_dict)
    .fillna(merged.get("Source_Occupation"))
    .fillna(merged.get("Source_Occupation_ONET"))
)

merged["Target_Occupation"] = (
    merged["Target_SOC"].map(soc_title_dict)
    .fillna(merged.get("Target_Occupation_Lightcast"))
    .fillna(merged.get("Target_Occupation_ONET"))
)

merged = merged.merge(
    ai_lookup.rename(columns={
        "SOC": "Source_SOC",
        "AI_Exposure_Group": "Source_AI_Exposure_Group"
    }),
    on="Source_SOC",
    how="left"
)

merged = merged.merge(
    ai_lookup.rename(columns={
        "SOC": "Target_SOC",
        "AI_Exposure_Group": "Target_AI_Exposure_Group"
    }),
    on="Target_SOC",
    how="left"
)

merged = merged[
    merged["Source_AI_Exposure_Group"].isin(SOURCE_EXPOSURE_GROUPS)
].copy()

merged = merged.merge(
    ai_economic.rename(columns={
        "SOC": "Target_SOC",
        "AI_Occupation_Title": "AI_Target_Occupation_Title",
        "AI_Exposure_Group": "AI_Target_Exposure_Group"
    }),
    on="Target_SOC",
    how="left"
)

merged["Preferred_Annual_Openings"] = merged["AI_Annual_Total_Openings"].combine_first(
    merged["LC_Target_2024_2025_Estimated_Annual_Openings"]
)

merged["Preferred_Growth_Rate"] = merged["AI_Growth_Rate"]
merged["Preferred_Annual_Percent_Change"] = merged["AI_Annual_Percent_Change"]

merged["Preferred_Median_Annual_Wage"] = merged["AI_50th_Percentile_Wage"].combine_first(
    merged["LC_Target_Median_Annual_Earnings"]
)

merged["Preferred_Education"] = merged["AI_Education_Value"]
merged["Preferred_Work_Experience"] = merged["AI_Work_Experience_Value"]

# ============================================================
# NORMALIZED SCORING AND RANKS
# UPDATED HYBRID LIGHTCAST + O*NET METHODOLOGY
# ============================================================

merged["Lightcast_Compat_Score"] = merged["Final_Compat_Index"] / 100

already_decimal = merged["Final_Compat_Index"].between(0, 1, inclusive="both")

merged.loc[
    already_decimal,
    "Lightcast_Compat_Score"
] = merged.loc[
    already_decimal,
    "Final_Compat_Index"
]

def rank_score_by_source(df, rank_col):
    max_rank = df.groupby("Source_SOC")[rank_col].transform("max")

    out = 1 - ((df[rank_col] - 1) / (max_rank - 1))

    out = out.where(max_rank > 1, 1)

    return out

merged["Lightcast_Rank_Score"] = rank_score_by_source(
    merged,
    "Lightcast_Rank"
)

merged["ONET_Rank_Score"] = rank_score_by_source(
    merged,
    "ONET_Rank"
)

merged["ONET_Tier_Score"] = merged["ONET_Tier"].map({
    "Primary-Short": 1.00,
    "Primary-Long": 0.70,
    "Supplemental": 0.35
})

merged["Lightcast_Combined_Score"] = (
    0.60 * merged["Lightcast_Compat_Score"] +
    0.40 * merged["Lightcast_Rank_Score"]
)

merged["ONET_Combined_Score"] = (
    0.70 * merged["ONET_Tier_Score"] +
    0.30 * merged["ONET_Rank_Score"]
)

def final_score(row):
    lc = row["Lightcast_Combined_Score"]
    onet_score = row["ONET_Combined_Score"]
    source_type = str(row["Source_Type"]).strip()

    if source_type == "Both" and pd.notna(lc) and pd.notna(onet_score):
        score = (0.60 * lc) + (0.40 * onet_score)
        score += 0.10
        return min(score, 1.00)

    if source_type == "Lightcast" and pd.notna(lc):
        return lc

    if source_type == "ONET" and pd.notna(onet_score):
        return onet_score

    if pd.notna(lc) and pd.notna(onet_score):
        return (0.60 * lc) + (0.40 * onet_score)

    if pd.notna(lc):
        return lc

    if pd.notna(onet_score):
        return onet_score

    return np.nan

merged["Both_Combined_Score"] = merged.apply(final_score, axis=1)

merged["Both_Combined_Score_100"] = (
    merged["Both_Combined_Score"] * 100
).round(1)

merged["Both_Combined_Rank"] = (
    merged.groupby("Source_SOC")["Both_Combined_Score"]
    .rank(method="first", ascending=False)
)

merged["Both_Average_Source_Rank"] = (
    merged[["Lightcast_Rank", "ONET_Rank"]]
    .mean(axis=1)
)

merged = merged.sort_values(
    [
        "Source_SOC",
        "Both_Combined_Rank",
        "Both_Combined_Score",
        "Target_SOC"
    ],
    ascending=[True, True, False, True],
    na_position="last"
).copy()

# ============================================================
# FINAL COLUMN ORDER
# ============================================================

first_cols = [
    "Source_SOC",
    "Source_Occupation",
    "Source_AI_Exposure_Group",
    "Target_SOC",
    "Target_Occupation",
    "Target_AI_Exposure_Group",
    "Source_Type",
    "Lightcast_Rank",
    "ONET_Rank",
    "Both_Average_Source_Rank",
    "Both_Combined_Rank",
    "Final_Compat_Index",
    "Lightcast_Compat_Score",
    "Lightcast_Rank_Score",
    "Lightcast_Combined_Score",
    "ONET_Tier",
    "ONET_Index",
    "ONET_Tier_Score",
    "ONET_Rank_Score",
    "ONET_Combined_Score",
    "Both_Combined_Score",
    "Both_Combined_Score_100",
    "Preferred_Median_Annual_Wage",
    "Preferred_Annual_Openings",
    "Preferred_Annual_Percent_Change",
    "Preferred_Growth_Rate",
    "Preferred_Education",
    "Preferred_Work_Experience",
    "AI_Annual_Total_Openings",
    "AI_Annual_Percent_Change",
    "AI_Education_Value",
    "AI_Work_Experience_Value",
    "AI_Growth_Rate",
    "AI_50th_Percentile_Wage",
    "LC_Target_Median_Hourly_Earnings",
    "LC_Target_Median_Annual_Earnings",
    "LC_Target_2024_Jobs",
    "LC_Target_2025_Jobs",
    "LC_Target_2024_2025_Change",
    "LC_Target_2024_2025_Estimated_Annual_Openings",
    "Source_ONET_SOC_Lightcast",
    "Target_ONET_SOC_Lightcast",
    "Source_ONET_SOC_ONET",
    "Target_ONET_SOC_ONET",
    "Lightcast_Source_File"
]

existing_first = [c for c in first_cols if c in merged.columns]
remaining = [c for c in merged.columns if c not in existing_first]

final = merged[existing_first + remaining].sort_values(
    ["Source_SOC", "Both_Combined_Rank", "Target_SOC"],
    na_position="last"
).copy()

score_summary = (
    final.groupby("Source_Type", dropna=False)
    .agg(
        Rows=("Target_SOC", "size"),
        Avg_Lightcast_Compat_Score=("Lightcast_Compat_Score", "mean"),
        Avg_ONET_Combined_Score=("ONET_Combined_Score", "mean"),
        Avg_Both_Combined_Score=("Both_Combined_Score", "mean"),
        Min_Both_Combined_Score=("Both_Combined_Score", "min"),
        Max_Both_Combined_Score=("Both_Combined_Score", "max")
    )
    .reset_index()
)

source_type_by_source = (
    final.groupby(["Source_SOC", "Source_Occupation", "Source_Type"])
    .size()
    .reset_index(name="Rows")
)

target_not_in_ai = (
    final[final["Target_AI_Exposure_Group"].isna()]
    .drop_duplicates(subset=["Target_SOC", "Target_Occupation"])
    .sort_values("Target_SOC")
)
# ============================================================
# CLEAN DIRECT LOWER-EXPOSURE SHEET
# High/Very High sources can lead to Medium, Low, or Very Low
# Medium sources can only lead to Low or Very Low
# ============================================================

def allowed_direct_transition(row):
    source_exp = row["Source_AI_Exposure_Group"]
    target_exp = row["Target_AI_Exposure_Group"]

    if source_exp in ["High", "Very High"]:
        return target_exp in HIGH_VERY_HIGH_TARGET_GROUPS

    if source_exp == "Medium":
        return target_exp in MEDIUM_TARGET_GROUPS

    return False

clean_direct_lower = final[
    final.apply(allowed_direct_transition, axis=1)
].copy()
# ============================================================
# FALLBACK: MAKE SURE EVERY HIGH / VERY HIGH SOURCE HAS
# AT LEAST ONE LOWER-EXPOSURE DIRECT RELATED OCCUPATION
# ============================================================

high_vh_sources = final[
    final["Source_AI_Exposure_Group"].isin(["High", "Very High"])
][["Source_SOC", "Source_Occupation", "Source_AI_Exposure_Group"]].drop_duplicates()

sources_with_direct = set(clean_direct_lower["Source_SOC"].astype(str).str.strip())

missing_sources = high_vh_sources[
    ~high_vh_sources["Source_SOC"].astype(str).str.strip().isin(sources_with_direct)
].copy()

fallback_pool = final[
    final["Source_SOC"].astype(str).str.strip().isin(
        missing_sources["Source_SOC"].astype(str).str.strip()
    )
].copy()

# Still require lower exposure only
fallback_pool = fallback_pool[
    fallback_pool["Target_AI_Exposure_Group"].isin(["Medium", "Low", "Very Low"])
].copy()

# Keep the best available lower-exposure option per missing source
fallback_direct = (
    fallback_pool
    .sort_values(
        [
            "Source_SOC",
            "Both_Combined_Score_100",
            "Both_Combined_Score",
            "Preferred_Median_Annual_Wage",
            "Preferred_Annual_Openings"
        ],
        ascending=[True, False, False, False, False],
        na_position="last"
    )
    .groupby("Source_SOC")
    .head(1)
    .copy()
)

fallback_direct["Fallback_Flag"] = "Fallback: best available lower-exposure option"

clean_direct_lower["Fallback_Flag"] = ""
clean_direct_lower = pd.concat(
    [clean_direct_lower, fallback_direct],
    ignore_index=True
)

print("High/Very High sources missing before fallback:", len(missing_sources))
print("Fallback rows added:", len(fallback_direct))
# ============================================================
# GATEWAY PATHWAYS
# Source occupation -> top 3 lower-exposure gateway occupations
# Gateway occupation -> additional lower-exposure destination occupations
# ============================================================

top_gateways = (
    clean_direct_lower
    .sort_values(
        ["Source_SOC", "Both_Combined_Rank", "Both_Combined_Score"],
        ascending=[True, True, False],
        na_position="last"
    )
    .groupby("Source_SOC")
    .head(3)
    .copy()
)

top_gateways["Gateway_Rank"] = top_gateways.groupby("Source_SOC").cumcount() + 1

# Load the full O*NET related occupations file again so gateways can be lower-exposure sources too
onet_gateway = pd.read_excel(ONET_FILE)
onet_gateway.columns = onet_gateway.columns.astype(str).str.strip()

onet_gateway = onet_gateway.rename(columns={
    "O*NET-SOC Code": "Gateway_ONET_SOC",
    "Title": "Gateway_Occupation_ONET",
    "Related O*NET-SOC Code": "Destination_ONET_SOC",
    "Related Title": "Destination_Occupation_ONET",
    "Relatedness Tier": "Destination_ONET_Tier",
    "Index": "Destination_ONET_Index"
})

onet_gateway["Gateway_ONET_SOC"] = onet_gateway["Gateway_ONET_SOC"].apply(fix_onet_detailed)
onet_gateway["Destination_ONET_SOC"] = onet_gateway["Destination_ONET_SOC"].apply(fix_onet_detailed)

onet_gateway["Gateway_SOC"] = onet_gateway.apply(
    lambda r: onet_to_soc(r["Gateway_ONET_SOC"], r["Gateway_Occupation_ONET"]),
    axis=1
)

onet_gateway["Destination_SOC"] = onet_gateway.apply(
    lambda r: onet_to_soc(r["Destination_ONET_SOC"], r["Destination_Occupation_ONET"]),
    axis=1
)

onet_gateway = onet_gateway.dropna(subset=["Gateway_SOC", "Destination_SOC"]).copy()

# Remove self-loops
onet_gateway = onet_gateway[
    onet_gateway["Gateway_SOC"] != onet_gateway["Destination_SOC"]
].copy()

onet_gateway["Destination_ONET_Index"] = onet_gateway["Destination_ONET_Index"].apply(clean_num)

onet_gateway["Destination_ONET_Tier_Score"] = onet_gateway["Destination_ONET_Tier"].map({
    "Primary-Short": 1.00,
    "Primary-Long": 0.75,
    "Supplemental": 0.50
})

tier_priority = {
    "Primary-Short": 1,
    "Primary-Long": 2,
    "Supplemental": 3
}

onet_gateway["Destination_ONET_Tier_Priority"] = (
    onet_gateway["Destination_ONET_Tier"].map(tier_priority)
)

onet_gateway = (
    onet_gateway
    .sort_values(
        [
            "Gateway_SOC",
            "Destination_SOC",
            "Destination_ONET_Tier_Priority",
            "Destination_ONET_Index"
        ],
        ascending=[True, True, True, True],
        na_position="last"
    )
    .drop_duplicates(subset=["Gateway_SOC", "Destination_SOC"], keep="first")
    .copy()
)

# Add destination exposure
onet_gateway = onet_gateway.merge(
    ai_lookup.rename(columns={
        "SOC": "Destination_SOC",
        "AI_Exposure_Group": "Destination_AI_Exposure_Group"
    }),
    on="Destination_SOC",
    how="left"
)

# Keep only possible lower-exposure gateway destinations.
# Final source-specific filtering happens after gateway paths are created.
onet_gateway = onet_gateway[
    onet_gateway["Destination_AI_Exposure_Group"].isin(["Very Low", "Low", "Medium"])
].copy()

# Add destination title and AI economic data
onet_gateway["Destination_Occupation"] = (
    onet_gateway["Destination_SOC"]
    .map(soc_title_dict)
    .fillna(onet_gateway["Destination_Occupation_ONET"])
)

onet_gateway = onet_gateway.merge(
    ai_economic.rename(columns={
        "SOC": "Destination_SOC",
        "AI_Occupation_Title": "AI_Destination_Occupation_Title",
        "AI_Exposure_Group": "AI_Destination_Exposure_Group"
    }),
    on="Destination_SOC",
    how="left"
)

# Rank gateway destinations within each gateway occupation
onet_gateway["Gateway_Destination_Rank"] = (
    onet_gateway
    .groupby("Gateway_SOC")["Destination_ONET_Index"]
    .rank(method="first", ascending=True)
)

# Join top gateways to their lower-exposure destinations
gateway_paths = top_gateways.merge(
    onet_gateway,
    left_on="Target_SOC",
    right_on="Gateway_SOC",
    how="left",
    suffixes=("", "_Gateway")
)

gateway_paths = gateway_paths[
    gateway_paths["Destination_SOC"].notna()
].copy()

# Final source-specific gateway filter:
# High/Very High sources can lead to Medium, Low, or Very Low.
# Medium sources can only lead to Low or Very Low.
def allowed_gateway_transition(row):
    source_exp = row["Source_AI_Exposure_Group"]
    dest_exp = row["Destination_AI_Exposure_Group"]

    if source_exp in ["High", "Very High"]:
        return dest_exp in HIGH_VERY_HIGH_TARGET_GROUPS

    if source_exp == "Medium":
        return dest_exp in MEDIUM_TARGET_GROUPS

    return False

gateway_paths = gateway_paths[
    gateway_paths.apply(allowed_gateway_transition, axis=1)
].copy()

# Remove cases where destination loops back to original source
gateway_paths = gateway_paths[
    gateway_paths["Destination_SOC"] != gateway_paths["Source_SOC"]
].copy()

# Remove cases where destination is same as gateway
gateway_paths = gateway_paths[
    gateway_paths["Destination_SOC"] != gateway_paths["Gateway_SOC"]
].copy()

gateway_paths["Pathway_Type"] = "Indirect / Gateway"

gateway_paths["Gateway_Occupation"] = gateway_paths["Target_Occupation"]
gateway_paths["Gateway_AI_Exposure_Group"] = gateway_paths["Target_AI_Exposure_Group"]

gateway_paths["Gateway_Pathway_Label"] = (
    gateway_paths["Source_Occupation"].astype(str)
    + " -> "
    + gateway_paths["Gateway_Occupation"].astype(str)
    + " -> "
    + gateway_paths["Destination_Occupation"].astype(str)
)

gateway_paths = gateway_paths.sort_values(
    [
        "Source_SOC",
        "Gateway_Rank",
        "Gateway_Destination_Rank",
        "Destination_ONET_Tier_Priority",
        "Destination_ONET_Index"
    ],
    ascending=[True, True, True, True, True],
    na_position="last"
).copy()

clean_gateway_paths = gateway_paths[[
    "Pathway_Type",
    "Gateway_Pathway_Label",

    "Source_SOC",
    "Source_Occupation",
    "Source_AI_Exposure_Group",

    "Gateway_Rank",
    "Gateway_SOC",
    "Gateway_Occupation",
    "Gateway_AI_Exposure_Group",
    "Source_Type",
    "Lightcast_Rank",
    "ONET_Rank",
    "Both_Combined_Rank",
    "Both_Combined_Score_100",

    "Destination_SOC",
    "Destination_Occupation",
    "Destination_AI_Exposure_Group",
    "Destination_ONET_Tier",
    "Destination_ONET_Index",
    "Destination_ONET_Tier_Score",
    "Gateway_Destination_Rank",

    "AI_Annual_Total_Openings",
    "AI_Annual_Percent_Change",
    "AI_Education_Value",
    "AI_Work_Experience_Value",
    "AI_Growth_Rate",
    "AI_50th_Percentile_Wage"
]].copy()

# ============================================================
# RELATED OCCUPATIONS FINAL SHEET
# Direct + Indirect pathways to Very Low, Low, and Medium only
# ============================================================

# ----------------------------
# DIRECT PATHWAYS
# Source occupation -> Related occupation
# ----------------------------

direct_related = clean_direct_lower.copy()

direct_related["Pathway_Type"] = "Direct"
direct_related["Gateway_SOC"] = ""
direct_related["Gateway_Occupation"] = ""
direct_related["Gateway_AI_Exposure_Group"] = ""
direct_related["Gateway_Rank"] = np.nan

# For direct rows, O*NET rank/index/tier describe Source -> Related Occupation
direct_related["Related_ONET_Rank"] = direct_related["ONET_Rank"]
direct_related["Related_ONET_Index"] = direct_related["ONET_Index"]
direct_related["Related_ONET_Tier"] = direct_related["ONET_Tier"]
direct_related["Related_ONET_Score"] = direct_related["ONET_Combined_Score"]

direct_related = direct_related.rename(columns={
    "Pathway_Type": "Pathway Type",
    "Source_SOC": "Source SOC",
    "Source_Occupation": "Source Occupation",
    "Source_AI_Exposure_Group": "Source AI Exposure",
    "Gateway_SOC": "Gateway SOC",
    "Gateway_Occupation": "Gateway Occupation",
    "Gateway_AI_Exposure_Group": "Gateway AI Exposure",
    "Gateway_Rank": "Gateway Rank",
    "Target_SOC": "Related Occupation SOC",
    "Target_Occupation": "Related Occupation",
    "Target_AI_Exposure_Group": "Related Occupation AI Exposure",
    "Source_Type": "Relationship Source",
    "Lightcast_Rank": "Lightcast Rank",
    "Both_Average_Source_Rank": "Average Source Rank",
    "Both_Combined_Rank": "Combined Rank",
    "Final_Compat_Index": "Lightcast Compatibility Index",
    "Lightcast_Combined_Score": "Lightcast Score",
    "Both_Combined_Score": "Combined Lightcast + O*NET Score",
    "Both_Combined_Score_100": "Combined Score 0-100",
    "Related_ONET_Rank": "Related Occupation O*NET Rank",
    "Related_ONET_Index": "Related Occupation O*NET Index",
    "Related_ONET_Tier": "Related Occupation O*NET Tier",
    "Related_ONET_Score": "Related Occupation O*NET Score"
})

direct_related = direct_related[[
    "Pathway Type",
    "Source SOC",
    "Source Occupation",
    "Source AI Exposure",
    "Gateway Rank",
    "Gateway SOC",
    "Gateway Occupation",
    "Gateway AI Exposure",
    "Related Occupation SOC",
    "Related Occupation",
    "Related Occupation AI Exposure",
    "Relationship Source",
    "Lightcast Rank",
    "Related Occupation O*NET Rank",
    "Average Source Rank",
    "Combined Rank",
    "Lightcast Compatibility Index",
    "Related Occupation O*NET Tier",
    "Related Occupation O*NET Index",
    "Lightcast Score",
    "Related Occupation O*NET Score",
    "Combined Lightcast + O*NET Score",
    "Combined Score 0-100"
]].copy()

# ----------------------------
# INDIRECT / GATEWAY PATHWAYS
# Source occupation -> Gateway occupation -> Related occupation
# ----------------------------

indirect_related = clean_gateway_paths.copy()

# For indirect rows, O*NET rank/index/tier describe Gateway -> Related Occupation
indirect_related["Related_ONET_Rank"] = indirect_related["Gateway_Destination_Rank"]
indirect_related["Related_ONET_Index"] = indirect_related["Destination_ONET_Index"]
indirect_related["Related_ONET_Tier"] = indirect_related["Destination_ONET_Tier"]
indirect_related["Related_ONET_Score"] = indirect_related["Destination_ONET_Tier_Score"]

# For indirect rows, the original source-to-gateway score is shown only as the gateway score
indirect_related["Source_to_Gateway_Combined_Rank"] = indirect_related["Both_Combined_Rank"]
indirect_related["Source_to_Gateway_Combined_Score_100"] = indirect_related["Both_Combined_Score_100"]

indirect_related = indirect_related.rename(columns={
    "Pathway_Type": "Pathway Type",
    "Source_SOC": "Source SOC",
    "Source_Occupation": "Source Occupation",
    "Source_AI_Exposure_Group": "Source AI Exposure",
    "Gateway_Rank": "Gateway Rank",
    "Gateway_SOC": "Gateway SOC",
    "Gateway_Occupation": "Gateway Occupation",
    "Gateway_AI_Exposure_Group": "Gateway AI Exposure",
    "Destination_SOC": "Related Occupation SOC",
    "Destination_Occupation": "Related Occupation",
    "Destination_AI_Exposure_Group": "Related Occupation AI Exposure",
    "Source_Type": "Relationship Source",
    "Related_ONET_Rank": "Related Occupation O*NET Rank",
    "Related_ONET_Index": "Related Occupation O*NET Index",
    "Related_ONET_Tier": "Related Occupation O*NET Tier",
    "Related_ONET_Score": "Related Occupation O*NET Score"
})

# For indirect rows, Lightcast Rank is intentionally blank because we are not ranking
# Gateway -> Related Occupation using Lightcast in this script.
indirect_related["Lightcast Rank"] = np.nan
indirect_related["Average Source Rank"] = np.nan
indirect_related["Combined Rank"] = indirect_related["Source_to_Gateway_Combined_Rank"]
indirect_related["Lightcast Compatibility Index"] = np.nan
indirect_related["Lightcast Score"] = np.nan
indirect_related["Combined Lightcast + O*NET Score"] = np.nan
indirect_related["Combined Score 0-100"] = indirect_related["Source_to_Gateway_Combined_Score_100"]

indirect_related = indirect_related[[
    "Pathway Type",
    "Source SOC",
    "Source Occupation",
    "Source AI Exposure",
    "Gateway Rank",
    "Gateway SOC",
    "Gateway Occupation",
    "Gateway AI Exposure",
    "Related Occupation SOC",
    "Related Occupation",
    "Related Occupation AI Exposure",
    "Relationship Source",
    "Lightcast Rank",
    "Related Occupation O*NET Rank",
    "Average Source Rank",
    "Combined Rank",
    "Lightcast Compatibility Index",
    "Related Occupation O*NET Tier",
    "Related Occupation O*NET Index",
    "Lightcast Score",
    "Related Occupation O*NET Score",
    "Combined Lightcast + O*NET Score",
    "Combined Score 0-100"
]].copy()

# ----------------------------
# COMBINE DIRECT + INDIRECT
# ----------------------------

related_occupations_final = pd.concat(
    [direct_related, indirect_related],
    ignore_index=True
)

# Direct pathways are always listed before gateway pathways.
related_occupations_final["Pathway Sort"] = related_occupations_final["Pathway Type"].map({
    "Direct": 1,
    "Indirect / Gateway": 2
})

related_occupations_final = related_occupations_final.sort_values(
    [
        "Source SOC",
        "Pathway Sort",
        "Combined Rank",
        "Gateway Rank",
        "Related Occupation O*NET Rank",
        "Combined Score 0-100",
        "Related Occupation SOC"
    ],
    ascending=[True, True, True, True, True, False, True],
    na_position="last"
).copy()

related_occupations_final = related_occupations_final.drop(
    columns=["Pathway Sort"],
    errors="ignore"
)

# ============================================================
# EXPORT
# ============================================================

with pd.ExcelWriter(OUT_FILE, engine="openpyxl") as writer:
    related_occupations_final.to_excel(writer, sheet_name="Related_Occs_Final", index=False)
    final.to_excel(writer, sheet_name="Final_Transitions", index=False)
    clean_direct_lower.to_excel(writer, sheet_name="Clean_Direct_Lower_Only", index=False)
    clean_gateway_paths.to_excel(writer, sheet_name="Clean_Gateway_Pathways", index=False)
    lc.to_excel(writer, sheet_name="Lightcast_Parsed", index=False)
    onet.to_excel(writer, sheet_name="ONET_Parsed", index=False)
    lc_raw.to_excel(writer, sheet_name="Lightcast_Raw_All", index=False)
    lc_source_econ.to_excel(writer, sheet_name="Lightcast_Source_Econ", index=False)
    lc_source_edu.to_excel(writer, sheet_name="Lightcast_Source_Edu", index=False)
    ai_economic.to_excel(writer, sheet_name="AI_Economic", index=False)
    score_summary.to_excel(writer, sheet_name="Score_Summary", index=False)
    source_type_by_source.to_excel(writer, sheet_name="Source_Type_By_Source", index=False)
    target_not_in_ai.to_excel(writer, sheet_name="Target_Not_In_AI", index=False)
    lc_errors.to_excel(writer, sheet_name="Lightcast_Parse_Errors", index=False)
    soc_title_lookup.to_excel(writer, sheet_name="SOC_Title_Lookup", index=False)
    crosswalk.to_excel(writer, sheet_name="Crosswalk_Used", index=False)

# ============================================================
# FORMAT WORKBOOK
# ============================================================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = load_workbook(OUT_FILE)

teal_fill = PatternFill("solid", fgColor="17C3B2")
light_fill = PatternFill("solid", fgColor="F2F2F2")
white_fill = PatternFill("solid", fgColor="FFFFFF")
white_font = Font(color="FFFFFF", bold=True)
body_font = Font(color="111111", size=10)
thin_border = Border(
    left=Side(style="thin", color="D9D9D9"),
    right=Side(style="thin", color="D9D9D9"),
    top=Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9")
)

for ws in wb.worksheets:
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    for cell in ws[1]:
        cell.fill = teal_fill
        cell.font = white_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = thin_border

    for row in ws.iter_rows(min_row=2):
        fill = light_fill if row[0].row % 2 == 0 else white_fill

        for cell in row:
            cell.fill = fill
            cell.font = body_font
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            cell.border = thin_border

    for col_idx, col_cells in enumerate(ws.columns, start=1):
        max_len = 0

        for cell in col_cells:
            if cell.value is not None:
                max_len = max(max_len, len(str(cell.value)))

        width = min(max(max_len + 2, 10), 42)
        ws.column_dimensions[get_column_letter(col_idx)].width = width

    ws.row_dimensions[1].height = 42

    for row_idx in range(2, ws.max_row + 1):
        ws.row_dimensions[row_idx].height = 24

# Specific formatting for final related occupations sheet
ws = wb["Related_Occs_Final"]

percent_cols = [
    "Lightcast Score",
    "Related Occupation O*NET Score",
    "Combined Lightcast + O*NET Score"
]

score_cols = [
    "Combined Score 0-100",
    "Lightcast Compatibility Index"
]

rank_cols = [
    "Gateway Rank",
    "Lightcast Rank",
    "Related Occupation O*NET Rank",
    "Average Source Rank",
    "Combined Rank",
    "Related Occupation O*NET Index"
]

for col_idx, cell in enumerate(ws[1], start=1):
    if cell.value in percent_cols:
        for row in range(2, ws.max_row + 1):
            ws.cell(row=row, column=col_idx).number_format = "0.0%"

    if cell.value in score_cols:
        for row in range(2, ws.max_row + 1):
            ws.cell(row=row, column=col_idx).number_format = "0.0"

    if cell.value in rank_cols:
        for row in range(2, ws.max_row + 1):
            ws.cell(row=row, column=col_idx).number_format = "0"

wb.save(OUT_FILE)

# ============================================================
# SAVE MAIN RELATED OCCUPATIONS SHEET AS CSV
# ============================================================

CSV_OUT_FILE = OUT_FILE.with_name(
    OUT_FILE.stem + "_Related_Occs_Final.csv"
)

related_occupations_final.to_csv(
    CSV_OUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("Saved CSV:")
print(CSV_OUT_FILE)

print("Saved rebuilt workbook:")
print(OUT_FILE)

print("\nRelated occupations final rows:", related_occupations_final.shape)
print("\nFinal rows:", final.shape)
print("\nSource type summary:")
print(final["Source_Type"].value_counts(dropna=False))

Crosswalk rows: 1016
Medium + High + Very High source SOCs: 519
Parsed Lightcast rows: 2665
Lightcast source SOCs: 270
O*NET rows: 9254
O*NET source SOCs: 460
High/Very High sources missing before fallback: 20
Fallback rows added: 0
Saved CSV:
C:\Users\301533\Documents\AI Analysis\Ligthcast Skills & RO\FINAL_REBUILT_High_VeryHigh_AND_Medium_Related_Occs_20260529_142926_Related_Occs_Final.csv
Saved rebuilt workbook:
C:\Users\301533\Documents\AI Analysis\Ligthcast Skills & RO\FINAL_REBUILT_High_VeryHigh_AND_Medium_Related_Occs_20260529_142926.xlsx

Related occupations final rows: (12931, 23)

Final rows: (10402, 50)

Source type summary:
Source_Type
ONET         7737
Both         1517
Lightcast    1148
Name: count, dtype: int64


In [3]:
missing_socs = missing_sources["Source_SOC"].unique()

check = final[
    final["Source_SOC"].isin(missing_socs)
].copy()

print("Missing source occupations:", len(missing_socs))

check.groupby(
    ["Source_SOC",
     "Source_Occupation",
     "Target_AI_Exposure_Group"]
).size().reset_index(name="Count")

Missing source occupations: 20


,Source_SOC,Source_Occupation,Target_AI_Exposure_Group,Count
0,13-1011,"Agents and Business Managers of Artists, Perfo...",High,8
1,13-1011,"Agents and Business Managers of Artists, Perfo...",Very High,16
2,13-2041,Credit Analysts,High,4
3,13-2041,Credit Analysts,Very High,17
4,13-2053,Insurance Underwriters,High,5
5,13-2053,Insurance Underwriters,Very High,18
6,13-2072,Loan Officers,High,5
7,13-2072,Loan Officers,Very High,16
8,13-2081,"Tax Examiners and Collectors, and Revenue Agents",High,3
9,13-2081,"Tax Examiners and Collectors, and Revenue Agents",Very High,20
